# Supporting Information

**Alpöge's Certified Lean Derivation Does Not Prove the Riemann Hypothesis: Granting Every Check, the Hypothesis Follows from a Premise Stronger Than Itself** (version 6)

Parker Emmerson (Yaohushuason), November 2026

This notebook reproduces every reading and every number that the note reports. It has seven parts.

1. **The archive.** The copy of the Lean archive that was read, its commit, the count of its files and lines, and the text scans of its sources: the words counted, the options set, the declarations of the names that the certified sentence is written with, the definitions whose whole body is `True`, and an index of the declarations on the route of Part 2.
2. **The route of the certified derivation.** The five theorems of `Main.lean`; the last two steps; the chain of named calls from the final theorem down to the windowed criterion of Landau type; the statement of that criterion and the library identities it calls; the three steps of the route that call a clause; and the conditions of the type `GeoP`, of which the route supplies the one named `anyGeometry`.
3. **The prover's core, version 4.33.1.** The definitions of `False`, of the counter-inscription and of "apart from", the three axioms, and the theorems `Classical.em` and `Classical.propComplete`.
4. **The Mathlib library at the commit to which the archive is pinned.** The definitions of the zeta function and of the hypothesis, the clauses that write a value where a rule withholds one, the clause for the integral, and the identities on the half-plane where the series converge.
5. **The note's own statements, checked by the prover**, and the numerical values the note quotes.
6. **The numbers that the note reports**, in one table.
7. **The readings added in version 3.** The published list of the names in the exported proof; the two halves of the windowed criterion and the lemma at which they meet; the division in the proof of Siegel's theorem; the degenerate model of the archive's geometries; the result of looking up each name that the note says is, or is not, in the exported proof; the four combinations of "the law holds" and "the rules are consistent"; the state of the verification record as read on 23 November 2026; and, added in version 6, the tie condition `GeoTiedP` that is in the exported proof and the model that fills it.

The scans are scans of text: each counts the files, or the places, in which a word occurs in the Lean sources, and each link of the chain of Part 2 is the occurrence of the name of one declaration in the text of another. They are reported as such. The prover is not run on the archive here; that the archive compiles and that its final theorems rest on three axioms is taken from the certification and the verification record cited in the note.

The Lean sources of the archive carry no licence file, so the notebook prints from them statement lines and definitions, and for proofs it prints counts of names and no proof text. The prover's core and the Mathlib library are distributed under the Apache 2.0 licence; the lines printed from them are quoted with their file names and line numbers.

In [1]:
import os, re, sys, json, math, shutil, subprocess, time
from collections import Counter, defaultdict
from pathlib import Path

ARCHIVE_URL    = "https://github.com/jsmorph/rh"                # a copy of the archive lean.tar.gz
ARCHIVE_COMMIT = "ecd09ff9b4aaf6c08f2b5ef17dc49ca9648a77de"
LEAN_TAG       = "v4.33.1"                                      # the prover's version
MATHLIB_COMMIT = "0df444a360eaa60ab8c11dca51a86af692955474"     # the library's commit

WORK    = Path.cwd()
ARCHIVE = WORK / "rh"
SRC     = ARCHIVE / "lean" / "src"
RESULT  = {}                      # every number that the note reports is collected here


def run(args, cwd=None, timeout=3600):
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return r.returncode, r.stdout, r.stderr


def fetch(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists() or dest.stat().st_size == 0:
        code_, _, err = run(["curl", "-sS", "-L", "--fail", "--max-time", "120", "-o", str(dest), url])
        if code_ != 0:
            raise RuntimeError("could not fetch " + url + ": " + err.strip())
    return dest


def show(path, pattern, before=0, after=0, label=None, limit=None, width=170):
    "Print the lines of a file that match a pattern, with their line numbers."
    lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    rx = re.compile(pattern)
    shown = 0
    for i, line in enumerate(lines):
        if rx.search(line):
            lo, hi = max(0, i - before), min(len(lines), i + after + 1)
            for j in range(lo, hi):
                print(f"{label or Path(path).name}:{j + 1}: {lines[j][:width]}")
            print()
            shown += 1
            if limit and shown >= limit:
                break
    if shown == 0:
        print(f"{label or Path(path).name}: no line matches {pattern!r}")


def word_rx(w):
    "A name counted where it is not continued, on either side, by a letter, a digit, an underscore or a prime."
    return re.compile(r"(?<![A-Za-z0-9_'])" + re.escape(w) + r"(?![A-Za-z0-9_'])")


def name_rx(w):
    "The same rule with letters and digits of every alphabet, subscripts included; used for the links of Part 2."
    return re.compile(r"(?<![\w'])" + re.escape(w) + r"(?![\w'])")


print("python", sys.version.split()[0])

python 3.13.18


## Part 1. The archive

The archive was released on 20 November 2026 as the file `lean.tar.gz` in fourteen parts. The copy read here is the repository named below, whose README records that the directory `lean/` holds the contents of the joined archive without modification, and gives the SHA-256 of the joined archive as `9631b4f0…49c7fc`, the value that the verification record cited in the note also gives.

In [2]:
if not SRC.exists():
    code_, out_, err_ = run(["git", "clone", "--depth", "1", ARCHIVE_URL, str(ARCHIVE)])
    print(err_.strip()[-300:])

code_, head, _ = run(["git", "-C", str(ARCHIVE), "rev-parse", "HEAD"])
head = head.strip()
print("repository :", ARCHIVE_URL)
print("commit     :", head)
print("commit is the one named above:", head == ARCHIVE_COMMIT)
print("lean-toolchain :", (ARCHIVE / "lean" / "lean-toolchain").read_text().strip())
manifest = json.loads((ARCHIVE / "lean" / "lake-manifest.json").read_text())
for pkg in manifest["packages"]:
    if pkg["name"] == "mathlib":
        print("mathlib rev    :", pkg["rev"])
        print("mathlib rev is the commit named above:", pkg["rev"] == MATHLIB_COMMIT)
        print("mathlib source :", pkg.get("url"), "| kind:", pkg.get("type"))
RESULT["archive_commit"] = head

repository : https://github.com/jsmorph/rh
commit     : ecd09ff9b4aaf6c08f2b5ef17dc49ca9648a77de
commit is the one named above: True
lean-toolchain : leanprover/lean4:v4.33.1
mathlib rev    : 0df444a360eaa60ab8c11dca51a86af692955474
mathlib rev is the commit named above: True
mathlib source : https://github.com/leanprover-community/mathlib4 | kind: git


### One pass over the Lean sources

The pass reads every file with the extension `.lean` under `lean/src`. It counts files, lines and bytes, and the occurrences of the words and strings listed. A *word* is counted where it is not continued, on either side, by a letter, a digit, an underscore or a prime, so that `riemannZeta` is counted apart from `riemannZeta_one_sub`. A *string* is counted wherever it occurs; the two library lemmas `Integrable.of_integral_ne_zero` and `differentiableAt_of_deriv_ne_zero` are counted by the strings `of_integral_ne_zero` and `of_deriv_ne_zero`.

The same pass collects six further things: the name of every option set with `set_option`; every line that declares one of the names with which the certified sentence is written; every theorem stated as `False` with no hypothesis; every definition whose whole body is the statement `True`, or a function with the value `True` at every argument; for the declarations on the route of Part 2, the file and line at which each is declared; and the list of the files that Part 2e reads.

In [3]:
WORDS = ["sorry", "admit", "axiom", "native_decide", "sorryAx", "implemented_by", "unsafe", "opaque",
         "by_contra", "exfalso", "absurd", "lt_trichotomy",
         "riemannZeta", "RiemannHypothesis", "riemannZeta_zero", "riemannZeta_one_sub",
         "div_zero", "inv_zero", "tsum_eq_zero_of_not_summable", "vonMangoldt",
         "integral_undef", "deriv_zero_of_not_differentiableAt", "anyGeometry", "run_cmd"]
STRINGS = ["eulerProduct", "Classical.", "ofReduceBool", "trustCompiler", "@[extern",
           "of_integral_ne_zero", "of_deriv_ne_zero", "skipKernelTC"]
# the files that Part 2e reads: those in which one of these strings occurs
GEO_TRIGGERS = ["anyGeometry", "geoCl", "GeoCl", "geoOf", "GeoOf", "GeoP", "oAt_", "parametersOfExtension"]

# the declarations on the route of Part 2
ROUTE_NAMES = ["riemannHypothesis", "rhOfAll", "allChi", "everyCharacter", "everyCharacter_of_row",
               "every_character_of_literature_of_vWin", "every_character_of_theta_integrals_local_matching",
               "every_character_of_green_v_win", "zero_free_all_of_green_v_win",
               "nonempty_hyps_definite_side_of_closed_bound", "hypotheses_nonempty_of_unramified_exponents_alt",
               "imprimitive", "zeroFreeAll_of_topG", "formB", "zeroFree_of_rpow_window", "lemmaL",
               "poleData", "poleData_char", "landau_window", "landau_window_key", "landau_tails",
               "landau_tails_chernoff", "landau_tails_integrable", "kernel_integral",
               "mellin_Gchi_of_ne_one", "mellin_Gchi_of_ne_one_ramp", "mellin_Gchi_of_ne_one_int", "mellin_ramp",
               "rh_of_every", "rh_of_right"]

WORD_RX = {w: word_rx(w) for w in WORDS}
DECL_RX = re.compile(
    r"^(?:@\[[^\]\n]*\]\s*)*(?:(?:private|protected|noncomputable|nonrec|unsafe|partial|scoped|local)\s+)*"
    r"(theorem|lemma|def|abbrev|structure|inductive|class|instance|opaque|axiom)\b", re.M)
SETOPT_RX = re.compile(r"\bset_option\s+([A-Za-z_][A-Za-z0-9_.]*)")
SENTENCE_NAMES = ["RiemannHypothesis", "riemannZeta", "hurwitzZetaEven", "completedHurwitzZetaEven", "Gammaℝ"]
SHADOW_RX = re.compile(
    r"^\s*(?:(?:private|protected|noncomputable)\s+)*(?:def|abbrev|theorem|lemma|instance|structure|inductive|class|axiom|opaque)\s+"
    r"(?:[^\s:(\[{]*\.)?(" + "|".join(SENTENCE_NAMES) + r")(?![\w'.])", re.M)
FALSE_RX = re.compile(r"^\s*(?:(?:private|protected)\s+)?(?:theorem|lemma)\s+\S+\s*:\s*False\s*(?::=|$)", re.M)
# a definition whose whole body is the statement True, or a function with the value True at every argument
TRUE_DEF_RX = re.compile(
    r"^(?:@\[[^\]\n]*\]\s*)*(?:(?:private|protected|noncomputable)\s+)*(def|abbrev)\s+(\S+)"
    r"((?:(?!:=)[^\n]|\n(?=[ \t]))*?):=\s*((?:fun\s[^\n=]*?=>\s*)*)True[ \t]*$", re.M)
HEAD_RX = re.compile(
    r"^(?:@\[[^\]\n]*\]\s*)*(?:(?:private|protected|noncomputable|nonrec)\s+)*"
    r"(?:theorem|lemma|def|abbrev)\s+(?:[^\s:(\[{]*\.)?(" + "|".join(re.escape(t) for t in ROUTE_NAMES) + r")(?![\w'])", re.M)

t0 = time.time()
files = sorted(SRC.rglob("*.lean"))
n_lines = n_bytes = 0
files_with, places, decl = Counter(), Counter(), Counter()
setopt_files, setopt_places = Counter(), Counter()
shadow, false_theorems, run_cmd_files, GEO_FILES, true_defs = [], [], [], [], []
INDEX = defaultdict(list)                       # name -> [(path, index of the line of the declaration)]
for p in files:
    b = p.read_bytes()
    n_bytes += len(b)
    t = b.decode("utf-8", errors="replace")
    n_lines += t.count("\n") + (1 if t and not t.endswith("\n") else 0)
    if any(k in t for k in GEO_TRIGGERS):
        GEO_FILES.append(p)
    for w, rx in WORD_RX.items():
        if w in t:
            c = len(rx.findall(t))
            if c:
                files_with[w] += 1
                places[w] += c
    for s in STRINGS:
        c = t.count(s)
        if c:
            files_with[s] += 1
            places[s] += c
    for m in DECL_RX.finditer(t):
        decl[m.group(1)] += 1
    if "set_option" in t:
        names = SETOPT_RX.findall(t)
        for k in set(names):
            setopt_files[k] += 1
        for k in names:
            setopt_places[k] += 1
    if any(k in t for k in SENTENCE_NAMES):
        for m in SHADOW_RX.finditer(t):
            shadow.append((str(p.relative_to(SRC)), t.count("\n", 0, m.start()) + 1, m.group(0).strip()[:160]))
    if "False" in t:
        for m in FALSE_RX.finditer(t):
            false_theorems.append((str(p.relative_to(SRC)), t.count("\n", 0, m.start()) + 1))
    if "True" in t:
        for m in TRUE_DEF_RX.finditer(t):
            rest = t[m.end():m.end() + 200].split("\n")
            if len(rest) > 1 and rest[1][:1] in (" ", "\t") and rest[1].strip():
                continue                                  # the body goes on in a further line
            true_defs.append((str(p.relative_to(SRC)), t.count("\n", 0, m.start()) + 1, m.group(2),
                              "function" if m.group(4).strip() else "statement"))
    if "run_cmd" in t and WORD_RX["run_cmd"].search(t):
        run_cmd_files.append(str(p.relative_to(SRC)))
    if any(k in t for k in ROUTE_NAMES):
        for m in HEAD_RX.finditer(t):
            INDEX[m.group(1)].append((p, t.count("\n", 0, m.start())))

print(f"Lean source files : {len(files):,}")
print(f"lines             : {n_lines:,}")
print(f"bytes             : {n_bytes:,}")
print(f"(pass took {time.time() - t0:.0f} s)")
print()
print(f"{'word or string':38s}{'files':>10s}{'places':>12s}")
for k in WORDS + STRINGS:
    print(f"{k:38s}{files_with[k]:>10,}{places[k]:>12,}")
print()
print("lines that open a declaration, by keyword:")
for k, v in decl.most_common():
    print(f"  {k:12s}{v:>10,}")
print(f"  theorem + lemma : {decl['theorem'] + decl['lemma']:,}")

RESULT.update(files=len(files), lines=n_lines, bytes=n_bytes,
              files_with=dict(files_with), places=dict(places), decl=dict(decl))

Lean source files : 175,187
lines             : 17,374,215
bytes             : 1,030,531,854
(pass took 58 s)

word or string                             files      places
sorry                                          0           0
admit                                          0           0
axiom                                          0           0
native_decide                                  0           0
sorryAx                                        0           0
implemented_by                                 0           0
unsafe                                         0           0
opaque                                         0           0
by_contra                                 19,531      31,278
exfalso                                    2,994       4,137
absurd                                    13,882      23,015
lt_trichotomy                              1,207       1,540
riemannZeta                                  141         710
RiemannHypothesis                  

### Options, the names of the certified sentence, and theorems stated as `False`

The first table lists every option that a file of the archive sets, with the number of files and of places. The options bound the resources of the elaborator (the part of the prover that turns source text into terms), configure how it searches and unfolds, or silence its style checks. The option `debug.skipKernelTC`, which suspends the kernel's check of a declaration, is counted in the pass above by the string `skipKernelTC`.

In [4]:
print(f"{'option':52s}{'files':>8s}{'places':>9s}")
for k, v in sorted(setopt_places.items(), key=lambda kv: -kv[1]):
    print(f"{k:52s}{setopt_files[k]:>8,}{v:>9,}")
print()
print("distinct options set:", len(setopt_places))
print("files containing the string skipKernelTC:", files_with["skipKernelTC"])
print()
print("lines declaring one of the names", ", ".join(SENTENCE_NAMES), ":", len(shadow))
for row in shadow[:10]:
    print("  ", row)
print("theorems stated as False with no hypothesis:", len(false_theorems))
for row in false_theorems[:10]:
    print("  ", row)
print()
print("files containing the command run_cmd:", len(run_cmd_files))
for f in run_cmd_files:
    text = (SRC / f).read_text(encoding="utf-8", errors="replace")
    n_lines_f = len([l for l in text.split("\n") if l.strip()])
    calls = Counter(re.findall(r"run_cmd\s+([A-Za-z_.]+)", text))
    ext = Counter(re.findall(r"([A-Za-z_.]+Ext\.addEntry)", text))
    print(f"   {f}: {n_lines_f} lines; command {dict(calls)}; it calls {dict(ext)}")
RESULT.update(setopt_distinct=len(setopt_places), skip_kernel=files_with["skipKernelTC"],
              shadow=len(shadow), false_theorems=len(false_theorems), run_cmd_files=len(run_cmd_files))

option                                                 files   places
maxHeartbeats                                         54,080   57,556
autoImplicit                                          10,743   11,203
maxSynthPendingDepth                                   7,478    7,478
synthInstance.maxHeartbeats                            6,477    7,381
backward.isDefEq.respectTransparency                   1,727    1,879
linter.unusedSectionVars                                 852      874
linter.unusedVariables                                   272      503
linter.unusedSimpArgs                                    113      272
genSizeOfSpec                                            148      181
genInjectivity                                           146      179
linter.style.haveILetI                                   104      122
backward.isDefEq.respectTransparency.types                62       87
synthInstance.maxSize                                     47       51
linter.dupNamespace 

### Definitions whose whole body is `True`

The cell lists the definitions that the pass found whose whole body is the statement `True`, or a function that has the value `True` at every argument. The search is a search of text: it finds a definition opened by `def` or `abbrev` whose text after the sign `:=` is `True`, or `fun … => True`, and nothing further. The condition `anyGeometry` of Part 2e is one of them.

In [5]:
print("definitions whose whole body is True, or a function with the value True at every argument:", len(true_defs))
print("of them functions:", sum(1 for r in true_defs if r[3] == "function"))
print()
print(f"{'name':40s}{'kind':11s}file and line")
for f, ln, name, kind in true_defs:
    print(f"{name:40s}{kind:11s}{f.split('/')[-1]}:{ln}")
RESULT["true_defs"] = [list(r) for r in true_defs]

definitions whose whole body is True, or a function with the value True at every argument: 17
of them functions: 9

name                                    kind       file and line
trivialClassCondition                   statement  OrderValuation.lean:14
trivialClassCondition                   function   HeckeValueIntegrality.lean:36
anyGeometry                             function   OverIx.lean:19
trivialTieCondition                     function   TwistedPlaceChoiceLiesOverPair.lean:133
SeedWtU.noCondition                     function   CuspFormLimitTwoPlaces.lean:17
TrivialRankCondition                    statement  NotInhAbsOfGamLe.lean:36
trivialCondition                        function   SecondInclusionProjection.lean:100
AlgClass.noCondition                    function   HasChartsOfForallIsAffineOpen.lean:146
trivialTieCondition                     function   FirstFormIffSecondForm.lean:155
GCond.noCondition                       function   GRepresents.lean:79
UnusedSlot55       

## Part 2. The route of the certified derivation

### 2a. The five theorems and the last two steps

`Main.lean` states five theorems. The first is stated with the Mathlib library's own definition `RiemannHypothesis`; the other four are stated with the definitions of `Statements.lean`, whose names are printed below.

In [6]:
show(SRC / "Main.lean", r"^(import|theorem)\b", label="Main.lean")
print("definitions of Statements.lean:")
for m in re.finditer(r"^def\s+(\S+)", (SRC / "Statements.lean").read_text(), re.M):
    print("  ", m.group(1))

Main.lean:1: import Statements

Main.lean:2: import Supplements.Kernels.MainTheoremsAll

Main.lean:4: theorem riemannHypothesis : RiemannHypothesis := rhOfAll

Main.lean:6: theorem riemannZeta_eq_zero_re : RiemannZetaZerosOnLine := rhOfAll

Main.lean:8: theorem LFunction_ne_zero_of_half_lt_re : DirichletLNonvanishingRightOfHalf := allChi

Main.lean:10: theorem LFunction_eq_zero_re : DirichletLZerosOnLine := claimAll.2.1

Main.lean:12: theorem LFunction_eq_zero_re_of_isPrimitive : PrimitiveDirichletLZerosOnLine := critAll

definitions of Statements.lean:
   RiemannZetaZerosOnLine
   DirichletLNonvanishingRightOfHalf
   DirichletLZerosOnLine
   PrimitiveDirichletLZerosOnLine


`riemannHypothesis` is `rhOfAll`, which applies `Formalism.VW.rh_of_every` to `allChi`, a derivation of the statement `UpperTheory.EveryCharacter`. The cell prints these statement lines, the definition of `EveryCharacter`, and, for the two last steps, the line numbers at which each named fact of the library is invoked.

In [7]:
K = SRC / "Supplements" / "Kernels" / "MainTheorems.lean"
show(K, r"^theorem (allChi|rhOfAll)\b", label="Supplements/Kernels/MainTheorems.lean")

E = SRC / "Development" / "AutomorphicArithmetic" / "CharacterBoxes" / "EveryCharacter.lean"
show(E, r"^def UpperTheory\.EveryCharacter\b", after=1, label="…/CharacterBoxes/EveryCharacter.lean")

G = SRC / "Formalism" / "GammaFactorNonvanishing"
R = G / "RhOfRight.lean"
C = G / "CritLine.lean"
show(R, r"^theorem Formalism\.VW\.rh_of_right\b", label="…/GammaFactorNonvanishing/RhOfRight.lean")
show(C, r"^theorem Formalism\.VW\.rh_of_every\b", label="…/GammaFactorNonvanishing/CritLine.lean")


def where(path, names):
    lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    out = {}
    for n in names:
        rx = word_rx(n)
        out[n] = [i + 1 for i, l in enumerate(lines) if rx.search(l)]
    return out, len(lines)


used_R, len_R = where(R, ["lt_trichotomy", "exfalso", "absurd", "riemannZeta_zero", "riemannZeta_one_sub",
                          "Complex.Gamma_ne_zero_of_re_pos", "Complex.cos_eq_zero_iff", "htriv", "hz"])
print(f"RhOfRight.lean has {len_R} lines; line numbers at which each name occurs:")
for k, v in used_R.items():
    print(f"  {k:34s}{v}")
lines_C = C.read_text(encoding="utf-8", errors="replace").splitlines()
start = next(i for i, l in enumerate(lines_C) if l.startswith("theorem Formalism.VW.rh_of_every"))
tail = "\n".join(lines_C[start:])
print()
print("in rh_of_every, the step from every character to the zeta function invokes:")
for n in ["Formalism.VW.rh_of_right", "DirichletCharacter ℂ 1", "LFunction_modOne_eq"]:
    print(f"  {n:34s}{'yes' if n in tail else 'no'}")
RESULT["last_step"] = {"RhOfRight_lines": len_R, "names": used_R}

Supplements/Kernels/MainTheorems.lean:7: theorem allChi : UpperTheory.EveryCharacter := UpperTheory.FinalTheorems.everyCharacter

Supplements/Kernels/MainTheorems.lean:9: theorem rhOfAll : RiemannHypothesis := Formalism.VW.rh_of_every allChi

…/CharacterBoxes/EveryCharacter.lean:13: def UpperTheory.EveryCharacter : Prop :=
…/CharacterBoxes/EveryCharacter.lean:14:   ∀ (N : ℕ) [NeZero N] (χ : DirichletCharacter ℂ N) (s : ℂ), 1 / 2 < s.re → (χ ≠ 1 ∨ s ≠ 1) → DirichletCharacter.LFunction χ s ≠ 0

…/GammaFactorNonvanishing/RhOfRight.lean:5: theorem Formalism.VW.rh_of_right (hz : ∀ s : ℂ, 1 / 2 < s.re → s ≠ 1 → riemannZeta s ≠ 0) : RiemannHypothesis := by

…/GammaFactorNonvanishing/CritLine.lean:29: theorem Formalism.VW.rh_of_every (h : UpperTheory.EveryCharacter) : RiemannHypothesis :=

RhOfRight.lean has 62 lines; line numbers at which each name occurs:
  lt_trichotomy                     [8, 44]
  exfalso                           [9]
  absurd                            [62]
  riemannZeta

### 2b. The chain of named calls

The cell follows the route from `riemannHypothesis` downward. A *link* is a pair of names: a declaration of the archive, and a name that occurs in the text of that declaration. The text of a declaration runs from the line that opens it to the line before the next line that opens a declaration or a section. For each link the cell prints the file and the lines of the declaration and the number of places at which the second name occurs in it. A link *holds* when that number is at least one.

The cell first prints, for each name of the route, the number of declarations of the archive that carry it. Where a name is carried by more than one declaration the cell takes the one in the file named here: `riemannHypothesis` in `Main.lean`; `formB` in `PrimitiveCosets/Imports.lean`, where it has the full name `UpperTheory.Comparison₂.formB` that `zeroFreeAll_of_topG` writes; and the two evaluations of integrals of Part 2d, `kernel_integral` and `mellin_ramp`, in the folder `LandauTheorems`, where the lemmas that call them stand.

In [8]:
NEXT_RX = re.compile(r"^(?:@\[|/--|/-!|theorem\b|lemma\b|def\b|abbrev\b|private\b|protected\b|noncomputable\b|instance\b|"
                     r"structure\b|inductive\b|class\b|namespace\b|end\b|section\b|open\b|variable\b|set_option\b|omit\b|"
                     r"include\b|universe\b|attribute\b|macro\b|syntax\b|elab\b|local\b|scoped\b|example\b)")
FILE_HINT = {"riemannHypothesis": "Main.lean", "formB": "PrimitiveCosets/Imports.lean",
             "kernel_integral": "LandauTheorems/kernelIntegral.lean", "mellin_ramp": "LandauTheorems/MellinRamp.lean"}


def declaration(name):
    "The file, the first and last line, and the text of the declaration of a name on the route."
    found = INDEX.get(name, [])
    if name in FILE_HINT:
        found = [(p, n) for p, n in found if str(p).endswith(FILE_HINT[name])]
    assert len(found) == 1, (name, len(found))
    p, n = found[0]
    lines = p.read_text(encoding="utf-8", errors="replace").split("\n")
    j = n + 1
    while j < len(lines) and not NEXT_RX.match(lines[j]):
        j += 1
    return p, n + 1, j, "\n".join(lines[n:j])


def statement(name, width=170, max_lines=12):
    "Print the statement of a declaration: its lines up to the sign := that opens the proof."
    p, a, b, text = declaration(name)
    cut = text.find(":=")
    st = text if cut < 0 else text[:cut]
    for k, line in enumerate(st.rstrip().split("\n")[:max_lines]):
        print(f"{p.name}:{a + k}: {line[:width]}")
    print()


LINKS = [
    ("riemannHypothesis", "rhOfAll"),
    ("rhOfAll", "rh_of_every"),
    ("rh_of_every", "rh_of_right"),
    ("rhOfAll", "allChi"),
    ("allChi", "everyCharacter"),
    ("everyCharacter", "everyCharacter_of_row"),
    ("everyCharacter_of_row", "every_character_of_literature_of_vWin"),
    ("every_character_of_literature_of_vWin", "every_character_of_theta_integrals_local_matching"),
    ("every_character_of_theta_integrals_local_matching", "every_character_of_green_v_win"),
    ("every_character_of_theta_integrals_local_matching", "nonempty_hyps_definite_side_of_closed_bound"),
    ("every_character_of_green_v_win", "imprimitive"),
    ("every_character_of_green_v_win", "zero_free_all_of_green_v_win"),
    ("zero_free_all_of_green_v_win", "zeroFreeAll_of_topG"),
    ("zeroFreeAll_of_topG", "level_gm_le_on_family_scales"),
    ("zeroFreeAll_of_topG", "formB"),
    ("formB", "zeroFree_of_rpow_window"),
    ("zeroFree_of_rpow_window", "lemmaL"),
    ("lemmaL", "landau_window"),
    ("landau_window", "landau_window_key"),
    ("landau_window_key", "landau_tails"),
    ("landau_tails", "landau_tails_chernoff"),
    ("landau_tails_chernoff", "landau_tails_integrable"),
    ("landau_tails_integrable", "integral_undef"),
    ("lemmaL", "poleData"),
    ("poleData", "poleData_char"),
    ("poleData_char", "mellin_Gchi_of_ne_one"),
    ("mellin_Gchi_of_ne_one", "mellin_Gchi_of_ne_one_ramp"),
    ("mellin_Gchi_of_ne_one_ramp", "mellin_Gchi_of_ne_one_int"),
    ("mellin_Gchi_of_ne_one_int", "of_integral_ne_zero"),
    ("nonempty_hyps_definite_side_of_closed_bound", "hypotheses_nonempty_of_unramified_exponents_alt"),
    ("nonempty_hyps_definite_side_of_closed_bound", "anyGeometry"),
    ("hypotheses_nonempty_of_unramified_exponents_alt", "anyGeometry"),
]

print("number of declarations in the archive with each name of the route:")
for k in ROUTE_NAMES:
    print(f"  {k:55s}{len(INDEX.get(k, []))}")
print()
print(f"{'declaration':52s}{'name occurring in its text':50s}{'places':>7s}   file and lines")
all_hold = True
link_rows = []
for a, b in LINKS:
    p, lo, hi, text = declaration(a)
    c = len(name_rx(b).findall(text))
    all_hold = all_hold and c >= 1
    link_rows.append((a, b, c, p.name, lo, hi))
    print(f"{a:52s}{b:50s}{c:>7d}   {p.name}:{lo}-{hi}")
print()
print("links:", len(LINKS), "| every link holds:", all_hold)
RESULT["route"] = {"links": len(LINKS), "all_hold": all_hold, "rows": link_rows}

number of declarations in the archive with each name of the route:
  riemannHypothesis                                      2
  rhOfAll                                                1
  allChi                                                 1
  everyCharacter                                         1
  everyCharacter_of_row                                  1
  every_character_of_literature_of_vWin                  1
  every_character_of_theta_integrals_local_matching      1
  every_character_of_green_v_win                         1
  zero_free_all_of_green_v_win                           1
  nonempty_hyps_definite_side_of_closed_bound            1
  hypotheses_nonempty_of_unramified_exponents_alt        1
  imprimitive                                            1
  zeroFreeAll_of_topG                                    1
  formB                                                  10
  zeroFree_of_rpow_window                                1
  lemmaL                                       

The files in which the declarations of the route stand:

In [9]:
seen = []
for a, _ in LINKS:
    p, lo, hi, _t = declaration(a)
    rel = str(p.relative_to(SRC))
    if (a, rel) not in seen:
        seen.append((a, rel))
        print(f"{a:52s}{rel}:{lo}")

riemannHypothesis                                   Main.lean:4
rhOfAll                                             Supplements/Kernels/MainTheorems.lean:9
rh_of_every                                         Formalism/GammaFactorNonvanishing/CritLine.lean:29
allChi                                              Supplements/Kernels/MainTheorems.lean:7
everyCharacter                                      Development/AutomorphicArithmetic/CharacterBoxes/MainConclusion.lean:10
everyCharacter_of_row                               Development/AutomorphicArithmetic/CharacterBoxes/MainConclusion.lean:7
every_character_of_literature_of_vWin               Development/AutomorphicArithmetic/ThetaCorrespondence/ThetaPairLifts/ThirdPlaceSeries/EveryCharacterOfLiteratureOfVWin.lean:810
every_character_of_theta_integrals_local_matching   Development/AutomorphicArithmetic/ThetaCorrespondence/ThetaPairLifts/ThirdPlaceSeries/SiegelEisensteinSecondTermHypotheses.lean:69
every_character_of_green_v_win         

### 2c. The windowed criterion of Landau type

The step `zeroFreeAll_of_topG` passes to the theorem `UpperTheory.Comparison₂.formB`, which is `Deduction.zeroFree_of_rpow_window`. The cell prints the definitions with which that theorem is stated, the theorem's statement, and the statements of the steps `imprimitive`, `zero_free_all_of_green_v_win` and `zeroFreeAll_of_topG`. It prints also the structure `PoleData`, which lists what the criterion asks of a function and of the set of its poles, and the statement of the theorem `landau_window`, which places every pole on or to the left of the line with real part `θ`.

It then reads the folder `LandauTheorems`, in which the theorem is proved. It lists every declaration of the folder with its text, records which declarations of the folder each one calls by name, and collects the declarations *reached* from `zeroFree_of_rpow_window` by such calls. For three groups of library facts it prints the declarations of the folder that call each fact and those among them that are reached: the identities on the half-plane where the series converge, the facts about the continued functions, and the lemmas about a value fixed by convention. Last it prints the reached declarations that open a proof by refutation with the command `by_contra`.

In [10]:
DL = SRC / "Development" / "AutomorphicArithmetic" / "DedekindLFunctions"
CF = DL / "CharFamily.lean"
for pat, aft in [(r"^def ramp\b", 1), (r"^def Gchi\b", 4), (r"^structure CharFamily\b", 8), (r"^def Gm\b", 0),
                 (r"^def ZeroFree\b", 1)]:
    show(CF, pat, after=aft, label="…/DedekindLFunctions/CharFamily.lean")
statement("zeroFree_of_rpow_window")
statement("imprimitive")
statement("zero_free_all_of_green_v_win")
statement("zeroFreeAll_of_topG")
show(DL / "LandauZeroFreeCriterion.lean", r"^structure PoleData\b", after=8, label="…/DedekindLFunctions/LandauZeroFreeCriterion.lean")
statement("landau_window")

LT = DL / "LandauTheorems"
lt_files = sorted(LT.glob("*.lean"))
lt_text = {p.name: p.read_text(encoding="utf-8", errors="replace") for p in lt_files}
lt_lines = sum(t.count("\n") + (1 if t and not t.endswith("\n") else 0) for t in lt_text.values())
print(f"folder LandauTheorems: {len(lt_files)} files, {lt_lines:,} lines")

# every declaration of the folder, with its text
ANY_HEAD = re.compile(r"^(?:@\[[^\]\n]*\]\s*)*(?:(?:private|protected|noncomputable|nonrec)\s+)*"
                      r"(?:theorem|lemma|def|abbrev)\s+([^\s:(\[{]+)")
folder_decl, repeated = {}, []
for fname, t in lt_text.items():
    lines = t.split("\n")
    i = 0
    while i < len(lines):
        m = ANY_HEAD.match(lines[i])
        if not m:
            i += 1
            continue
        short = m.group(1).split(".")[-1]
        j = i + 1
        while j < len(lines) and not NEXT_RX.match(lines[j]):
            j += 1
        if short in folder_decl:
            repeated.append(short)
        folder_decl[short] = (fname, i + 1, j, "\n".join(lines[i:j]))
        i = j
print(f"declarations in the folder: {len(folder_decl)}; names carried by more than one of them: {len(set(repeated))}")

# the calls by name inside the folder, and what is reached from the windowed criterion
calls = {a: {b for b in folder_decl if b != a and b in text and name_rx(b).search(text)}
         for a, (_f, _lo, _hi, text) in folder_decl.items()}
START = "zeroFree_of_rpow_window"
reached, queue = {START}, [START]
while queue:
    a = queue.pop()
    for b in calls[a]:
        if b not in reached:
            reached.add(b)
            queue.append(b)
print(f"declarations reached from {START} by calls by name: {len(reached)}")
print(f"files in which they stand: {len({folder_decl[a][0] for a in reached})}")
print()


def callers(name):
    users = sorted(a for a, (_f, _lo, _hi, text) in folder_decl.items() if name_rx(name).search(text))
    return users, [a for a in users if a in reached]


SERIES = ["LSeries_twist_vonMangoldt_eq", "LSeries_vonMangoldt_eq_deriv_riemannZeta_div", "LFunction_eq_LSeries",
          "deriv_LFunction_eq_deriv_LSeries"]
CONTINUED = ["differentiable_LFunction", "differentiableAt_riemannZeta", "riemannZeta_residue_one",
             "LFunction_ne_zero_of_one_le_re", "riemannZeta_ne_zero_of_one_le_re"]
CONV = ["integral_undef", "of_integral_ne_zero", "deriv_zero_of_not_differentiableAt", "div_zero", "inv_zero",
        "tsum_eq_zero_of_not_summable"]
landau_calls = {}
for title, group in [("library identities on the half-plane where the series converge", SERIES),
                     ("library facts about the continued functions", CONTINUED),
                     ("library lemmas about a value fixed by convention", CONV)]:
    print(title + " — declarations of the folder that call each, and those among them that are reached:")
    for n in group:
        users, r = callers(n)
        landau_calls[n] = {"callers": users, "reached": r}
        print(f"  {n:46s}{len(users):>2d}   reached: {r}")
    print()
bc_users, bc_reached = callers("by_contra")
bcases_users, bcases_reached = callers("by_cases")
print(f"the command by_contra: {len(bc_users)} declarations of the folder, {len(bc_reached)} of them reached:")
print("  ", bc_reached)
print(f"the command by_cases : {len(bcases_users)} declarations of the folder, {len(bcases_reached)} of them reached")
RESULT["landau"] = {"files": len(lt_files), "lines": lt_lines, "declarations": len(folder_decl),
                    "reached": len(reached), "reached_files": len({folder_decl[a][0] for a in reached}),
                    "calls": landau_calls, "by_contra_reached": bc_reached, "by_cases_reached": len(bcases_reached)}

…/DedekindLFunctions/CharFamily.lean:10: def ramp (f : ℕ → ℂ) (y : ℝ) : ℂ :=
…/DedekindLFunctions/CharFamily.lean:11:   ∑ n ∈ Finset.range (⌊y⌋₊ + 1), (Λ n : ℂ) * f n * ((max (y / n - 1) 0 : ℝ) : ℂ)

…/DedekindLFunctions/CharFamily.lean:15: def Gchi {N : ℕ} [NeZero N] (χ : DirichletCharacter ℂ N) (y : ℝ) : ℂ :=
…/DedekindLFunctions/CharFamily.lean:16:   if χ = 1 then
…/DedekindLFunctions/CharFamily.lean:17:     ramp (fun n => χ n) y - ((y * Real.log y : ℝ) : ℂ) + (((1 + Real.eulerMascheroniConstant) * y : ℝ) : ℂ)
…/DedekindLFunctions/CharFamily.lean:18:   else
…/DedekindLFunctions/CharFamily.lean:19:     ramp (fun n => χ n) y + (y : ℂ) * (deriv (DirichletCharacter.LFunction χ) 1 / DirichletCharacter.LFunction χ 1)

…/DedekindLFunctions/CharFamily.lean:24: structure CharFamily where
…/DedekindLFunctions/CharFamily.lean:25:   ι : Type
…/DedekindLFunctions/CharFamily.lean:26:   [fin : Fintype ι]
…/DedekindLFunctions/CharFamily.lean:27:   N : ι → ℕ
…/DedekindLFunctions/CharFamily.lean:28: 

### 2d. Three steps of the route that call a lemma about a value fixed by convention

Two lemmas of the route conclude that a function is integrable from the clause for the integral. The cell prints their statements and the statements of the two evaluations of integrals on which they rest, and, for the proof of each lemma, the names from a fixed list that occur in it with the number of places. No proof text is printed. Part 2b shows that both lemmas lie on the route; Part 4 prints the library's clause and the lemma that reads it backwards.

A third lemma, `deriv_conj_comp_conj`, is stated for every function of a complex variable, differentiable or not. The cell prints its statement, the names `by_cases` and `deriv_zero_of_not_differentiableAt` with the number of places at which they occur in its proof, and whether Part 2c reached it from the windowed criterion.

In [11]:
for n in ["kernel_integral", "landau_tails_integrable", "mellin_ramp", "mellin_Gchi_of_ne_one_int"]:
    statement(n)

PROOF_NAMES = ["by_contra", "integral_undef", "kernel_integral", "positivity", "linarith",
               "of_integral_ne_zero", "mellin_ramp", "div_ne_zero", "Integrable.mono", "integrable_of"]
clause_steps = {}
for n in ["landau_tails_integrable", "mellin_Gchi_of_ne_one_int"]:
    p, lo, hi, text = declaration(n)
    proof = text[text.find(":="):]
    used = {k: len(word_rx(k).findall(proof)) for k in PROOF_NAMES if word_rx(k).search(proof)}
    clause_steps[n] = {"file": p.name, "lines": [lo, hi], "names": used}
    print(f"{n}  ({p.name}:{lo}-{hi}); names in its proof: {used}")
print()
fname, lo, hi, text = folder_decl["deriv_conj_comp_conj"]
cut = text.find(":=")
for k, line in enumerate(text[:cut].rstrip().split("\n")):
    print(f"{fname}:{lo + k}: {line[:170]}")
proof = text[cut:]
used = {k: len(name_rx(k).findall(proof)) for k in ["by_cases", "deriv_zero_of_not_differentiableAt"] if name_rx(k).search(proof)}
clause_steps["deriv_conj_comp_conj"] = {"file": fname, "lines": [lo, hi], "names": used,
                                        "reached": "deriv_conj_comp_conj" in reached,
                                        "called_by": sorted(a for a in folder_decl if "deriv_conj_comp_conj" in calls[a])}
print(f"deriv_conj_comp_conj  ({fname}:{lo}-{hi}); names in its proof: {used}")
print("   reached from the windowed criterion:", clause_steps["deriv_conj_comp_conj"]["reached"],
      "| called by:", clause_steps["deriv_conj_comp_conj"]["called_by"])
print()
print("in the whole archive:")
for k in ["of_integral_ne_zero", "integral_undef", "of_deriv_ne_zero", "deriv_zero_of_not_differentiableAt"]:
    print(f"  {k:40s}{places[k]:>5,} places in {files_with[k]:>4,} files")
RESULT["clause_steps"] = clause_steps

kernelIntegral.lean:9: theorem kernel_integral (k : ℕ) (a : ℝ) (ha : 0 < a) :
kernelIntegral.lean:10:     ∫ u in Ioi (0 : ℝ), u ^ k * Real.exp (-a * u) / (k.factorial : ℝ) = (a ^ (k + 1))⁻¹

landauTails.lean:10: lemma landau_tails_integrable (k : ℕ) {a : ℝ} (ha : 0 < a) :
landauTails.lean:11:     IntegrableOn (fun u : ℝ => u ^ k * Real.exp (-a * u) / (k.factorial : ℝ)) (Ioi 0)

MellinRamp.lean:10: theorem mellin_ramp (n : ℕ) (hn : 1 ≤ n) (s : ℂ) (hs : 1 < s.re) :
MellinRamp.lean:11:     ∫ y in Ioi (1 : ℝ), ((max (y / n - 1) 0 : ℝ) : ℂ) * (y : ℂ) ^ (-s - 1) = (n : ℂ) ^ (-s) / (s * (s - 1))

mellinGchiOfNeOne.lean:19: theorem mellin_Gchi_of_ne_one_int (n : ℕ) (hn : 1 ≤ n) (s : ℂ) (hs : 1 < s.re) :
mellinGchiOfNeOne.lean:20:     IntegrableOn (fun y : ℝ => ((max (y / n - 1) 0 : ℝ) : ℂ) * (y : ℂ) ^ (-s - 1)) (Ioi 1)

landau_tails_integrable  (landauTails.lean:10-17); names in its proof: {'by_contra': 1, 'integral_undef': 1, 'kernel_integral': 1, 'positivity': 1, 'linarith': 1}
mellin_Gchi_o

### 2e. The conditions of the type `GeoP`

The archive defines a type `GeoP` of conditions on its objects. The cell prints the type; the three declarations whose stated type it is, `geoOf`, `anyGeometry` and `geoCl`; the record `OAt`, whose field `geo` is a derivation of a condition `Γ` at an object, and the component `OX` built from that record; the structure `Parameters` of the parameters of a family, whose last field is a condition, with its projection `Parameters.s`, which forms the component `OX` from that condition; and the constructor `parametersOfExtension`, whose third argument is the condition, with the definition `parametersOfDependence` that calls it.

It then reads the files of the archive in which a name of this group occurs (the list was collected in Part 1) and reports, as counts of text:

* the declarations whose stated type is `GeoP`;
* the declarations that carry the short name `geoOf`, of which there are three, with three different types; and, for every place of the word `geoOf`, the argument that follows it. The term of type `GeoP` takes a family `Q` as its argument, and its places are those at which `Q` follows;
* the declarations of the file `NotLamAreShift.lean` whose text names `geoOf`, with the number of places and of files of the archive at which each of them is named;
* the argument that follows the closure `geoCl` at each of its places, and where each place of `anyGeometry` stands;
* the argument in which a condition is supplied to `OAt`, to `OX` and to `parametersOfExtension`, at each of their places in these files.

Last it counts the places of `geoCl anyGeometry` in the two declarations of the route that assemble the bundle of hypotheses: in the statement of `hypotheses_nonempty_of_unramified_exponents_alt`, the lines before the sign `:=`, with the name of the condition that each of the two premises carries; and in the whole text of `nonempty_hyps_definite_side_of_closed_bound`. Of statements the cell prints names and counts, and of proofs it prints counts.

In [12]:
AM = SRC / "Development" / "AbelianVarietyTheory" / "Definitions" / "AutomorphicMembership"
TP = SRC / "Development" / "AutomorphicArithmetic" / "ThetaCorrespondence" / "ThetaPairLifts" / "ThirdPlaceSeries"
show(AM / "OverIx.lean", r"^abbrev GeoP\b", label="…/AutomorphicMembership/OverIx.lean")
show(AM / "OverIx.lean", r"^def geoOf\b", after=1, label="…/AutomorphicMembership/OverIx.lean")
show(AM / "OverIx.lean", r"^def anyGeometry\b", label="…/AutomorphicMembership/OverIx.lean")
show(AM / "OverIx.lean", r"^structure OAt\b", after=3, label="…/AutomorphicMembership/OverIx.lean")
show(AM / "OverIx.lean", r"^def OX\b", label="…/AutomorphicMembership/OverIx.lean")
show(AM / "NotGeoClFalse.lean", r"^inductive GeoCl\b", after=1, label="…/AutomorphicMembership/NotGeoClFalse.lean")
show(AM / "NotGeoClFalse.lean", r"^def geoCl\b", label="…/AutomorphicMembership/NotGeoClFalse.lean")
show(TP / "CoreAltAltBis.lean", r"^structure Parameters\b", after=4, label="…/ThirdPlaceSeries/CoreAltAltBis.lean")
show(TP / "CoreAltAltBis.lean", r"def Parameters\.s\b", label="…/ThirdPlaceSeries/CoreAltAltBis.lean", width=260)
show(TP / "CoreAltAltBis.lean", r"def parametersOfExtension\b", label="…/ThirdPlaceSeries/CoreAltAltBis.lean", width=230)
show(TP / "Prm1.lean", r"def parametersOfDependence\b", after=1, label="…/ThirdPlaceSeries/Prm1.lean", width=230)

…/AutomorphicMembership/OverIx.lean:14: abbrev GeoP : Type 1 := ∀ D : D7, Formalism.L2.Pj.Core D → DivX D → Prop

…/AutomorphicMembership/OverIx.lean:16: def geoOf (Q : ∀ D : D7, GlobalTheory.ShFam.KitClass D) : GeoP := fun D J X =>
…/AutomorphicMembership/OverIx.lean:17:   (∃ c : ℕ, 0 < c ∧ GlobalTheory.ShFam.IsGeoOf (Q D) J c X) ∧ GlobalTheory.ShFam.Pad.HasTrueSurf X

…/AutomorphicMembership/OverIx.lean:19: def anyGeometry : GeoP := fun _ _ _ => True

…/AutomorphicMembership/OverIx.lean:44: structure OAt (Γ : GeoP) {w : ColW B f T} {Z : w.Lt} {Wt : Set B.places.Pl} {p0 : BaseLevelRowSide (w.D Z)} (t : RowsT K w Z Wt p0)
…/AutomorphicMembership/OverIx.lean:45:     (X : DivX (w.D Z)) : Type 1 where
…/AutomorphicMembership/OverIx.lean:46:   geo : Γ (w.st Z).toD7 t.r.z.core.toCore X
…/AutomorphicMembership/OverIx.lean:47:   lam : LamAre t.r.Z X

…/AutomorphicMembership/OverIx.lean:49: def OX (Γ : GeoP) : OComponent₂ := fun _ _ _ _ _ _ _ O t => OAt Γ t O.X

…/AutomorphicMembership/NotGeoC

In [13]:
geo_text = {p: p.read_text(encoding="utf-8", errors="replace") for p in GEO_FILES}
print("files of the archive in which a name of this group occurs:", len(geo_text))
print()


def line_of(t, pos):
    return t.count("\n", 0, pos) + 1


def read_arg(text, i):
    "The argument that begins at or after the position i: a group in parentheses, or a run of characters up to a space or a bracket."
    n = len(text)
    while i < n and text[i].isspace():
        i += 1
    if i < n and text[i] == "(":
        depth, j = 0, i
        while j < n:
            if text[j] == "(":
                depth += 1
            elif text[j] == ")":
                depth -= 1
                if depth == 0:
                    return " ".join(text[i:j + 1].split()), j + 1
            j += 1
        return text[i:i + 40], n
    m = re.match(r"[^\s()\[\]{},⟨⟩]+", text[i:])
    if not m:
        return text[i:i + 1], i + 1
    return m.group(0), i + m.end()


# the declarations whose stated type is GeoP
TYPED = re.compile(r"^(?:@\[[^\]\n]*\]\s*)*(?:(?:private|protected|noncomputable)\s+)*(?:def|abbrev|theorem|lemma)\s+(\S+)[^\n]*:\s*GeoP\s*:=", re.M)
typed = sorted((m.group(1), p.name, line_of(t, m.start())) for p, t in geo_text.items() for m in TYPED.finditer(t))
print("declarations whose stated type is GeoP:")
for row in typed:
    print("  ", row)
print()

# the declarations that carry the short name geoOf
NAMED = re.compile(r"^(?:@\[[^\]\n]*\]\s*)*(?:(?:private|protected|noncomputable)\s+)*(theorem|lemma|def|abbrev)\s+geoOf(?![\w'])([^\n]*)", re.M)
named = []
for p, t in geo_text.items():
    for m in NAMED.finditer(t):
        ty = re.search(r"\)\s*:\s*([^:=]+?)\s*(?::=.*|where\s*)$", m.group(2))
        named.append((p.name, line_of(t, m.start()), m.group(1), ty.group(1) if ty else "?"))
named.sort()
print("declarations with the short name geoOf, each with its stated type:")
for row in named:
    print("  ", row)
print()

rows = sorted((p.name, line_of(t, m.start()), read_arg(t, m.end())[0][:14])
              for p, t in geo_text.items() for m in name_rx("geoOf").finditer(t))
print("every place of the word geoOf, with the argument that follows it:")
for r in rows:
    print("  ", r)
member = [r for r in rows if r[2].lstrip("(").startswith("Q")]
member_files = sorted({r[0] for r in member})
print(f"places of the word geoOf: {len(rows)} in {len({r[0] for r in rows})} files")
print(f"places at which the family Q follows (the term of type GeoP): {len(member)} in {len(member_files)} files: {member_files}")
print()

# the lemmas about the term geoOf, and where they are named
lem_path = AM / "NotLamAreShift.lean"
lem_lines = geo_text[lem_path].split("\n")
lem_decl, i = {}, 0
while i < len(lem_lines):
    m = ANY_HEAD.match(lem_lines[i])
    if not m:
        i += 1
        continue
    j = i + 1
    while j < len(lem_lines) and not NEXT_RX.match(lem_lines[j]):
        j += 1
    lem_decl[m.group(1)] = "\n".join(lem_lines[i:j])
    i = j
about = [n for n, text in lem_decl.items() if name_rx("geoOf").search(text)]
print(f"NotLamAreShift.lean: {len(lem_decl)} declarations; those whose text names geoOf: {about}")
lemma_use = {}
for n in about:
    pl_ = sum(len(name_rx(n).findall(t)) for t in geo_text.values())
    fl_ = sum(1 for t in geo_text.values() if name_rx(n).search(t))
    lemma_use[n] = [pl_, fl_]
    print(f"   {n:12s} is named at {pl_} place(s) in {fl_} file(s) of the archive")
print("   places in that file at which an attribute is set on a declaration ('@['):", geo_text[lem_path].count("@["))
print()

# what follows the closure, and where anyGeometry stands
after_cl = Counter(read_arg(t, m.end())[0] for t in geo_text.values() for m in name_rx("geoCl").finditer(t))
print(f"the argument that follows geoCl, at its {sum(after_cl.values())} places:")
for a, v in after_cl.most_common():
    print(f"   {v:5d}  {a}")
where_any, lemmas_any, any_files = Counter(), Counter(), set()
for p, t in geo_text.items():
    for m in name_rx("anyGeometry").finditer(t):
        any_files.add(p)
        before = t[:m.start()].rstrip()
        head = t[t.rfind("\n", 0, m.start()) + 1:m.start()]
        if before.endswith("geoCl"):
            key = "in geoCl anyGeometry"
        elif before.endswith("OX"):
            key = "in OX anyGeometry"
        elif before.endswith("OAt"):
            key = "in OAt anyGeometry"
        elif re.match(r"\s*def\s*$", head):
            key = "its definition"
        else:
            key = "the last argument of a lemma"
            f = re.findall(r"[\w.']*[_.][\w.']*", head)
            lemmas_any[f[-1].split(".")[-1] if f else "?"] += 1
        where_any[key] += 1
print(f"anyGeometry: {sum(where_any.values())} places in {len(any_files)} files:")
for a, v in where_any.most_common():
    print(f"   {v:5d}  {a}")
print("   the lemmas that take it as their last argument:", dict(lemmas_any))
print()

# the argument in which a condition is supplied
entry = {}
for nm, k in [("OAt", 1), ("OX", 1), ("parametersOfExtension", 3)]:
    c = Counter()
    for t in geo_text.values():
        for m in name_rx(nm).finditer(t):
            j, a = m.end(), ""
            for _ in range(k):
                a, j = read_arg(t, j)
            c[a if len(a) <= 46 else a[:46] + "…"] += 1
    entry[nm] = dict(c)
    print(f"argument {k} of {nm}, at its {sum(c.values())} places:")
    for a, v in c.most_common():
        print(f"   {v:5d}  {a}")
print()

# the two declarations of the route
geo = {"files_read": len(geo_text), "typed": typed, "named": named,
       "geoOf_places": [len(rows), len({r[0] for r in rows})], "member": [len(member), len(member_files)],
       "lemmas_about": lemma_use, "after_geoCl": dict(after_cl), "where_any": dict(where_any),
       "any_places_files": [sum(where_any.values()), len(any_files)], "lemmas_any": dict(lemmas_any), "entry": entry}
p, lo, hi, text = declaration("hypotheses_nonempty_of_unramified_exponents_alt")
st = text[:text.find(":= by")] if ":= by" in text else text[:text.find(":=")]
carried = [re.match(r"\s*([\w.]+)", st[m.end():]).group(1).split(".")[-1] for m in re.finditer(r"geoCl anyGeometry\)\)", st)]
geo["consumer"] = {"file": p.name, "lines": [lo, hi], "statement_geoCl_anyGeometry": st.count("geoCl anyGeometry"),
                   "carried": carried, "parametersOfDependence": len(name_rx("parametersOfDependence").findall(st)),
                   "statement_geoOf": len(name_rx("geoOf").findall(st)),
                   "concludes_Nonempty_HypsAlt": "Nonempty (_root_.GlobalTheory.Fc.HypsAlt" in st}
print("hypotheses_nonempty_of_unramified_exponents_alt, statement:")
print("   places of 'geoCl anyGeometry':", geo["consumer"]["statement_geoCl_anyGeometry"])
print("   the conditions that the two premises carry:", carried)
print("   places of 'parametersOfDependence' (a third premise):", geo["consumer"]["parametersOfDependence"])
print("   places of 'geoOf':", geo["consumer"]["statement_geoOf"])
print("   the statement concludes 'Nonempty (… HypsAlt …)':", geo["consumer"]["concludes_Nonempty_HypsAlt"])
p, lo, hi, text = declaration("nonempty_hyps_definite_side_of_closed_bound")
geo["supplier"] = {"file": p.name, "lines": [lo, hi], "geoCl_anyGeometry": text.count("geoCl anyGeometry"),
                   "geoOf": len(name_rx("geoOf").findall(text))}
print("nonempty_hyps_definite_side_of_closed_bound, whole text:")
print("   places of 'geoCl anyGeometry':", geo["supplier"]["geoCl_anyGeometry"])
print("   places of 'geoOf':", geo["supplier"]["geoOf"])
RESULT["geo"] = geo

files of the archive in which a name of this group occurs: 488



declarations whose stated type is GeoP:
   ('anyGeometry', 'OverIx.lean', 19)
   ('geoCl', 'NotGeoClFalse.lean', 26)
   ('geoOf', 'OverIx.lean', 16)



declarations with the short name geoOf, each with its stated type:
   ('AffineOpenChartCompStructureEqSpMap.lean', 503, 'theorem', 'Geo (Y Z.i) Z.π')
   ('OverIx.lean', 16, 'def', 'GeoP')
   ('RingKrullDimIntegers.lean', 600, 'def', 'DV.Geo F')



every place of the word geoOf, with the argument that follows it:
   ('AffineOpenChartCompStructureEqSpMap.lean', 503, '(Z : Mdl Y i₀)')
   ('AtlasModelComparison.lean', 50, 'Z')
   ('AtlasModelComparison.lean', 53, 'Z')
   ('ExistsDeRhamFlag.lean', 69, 'Z')
   ('ExistsFlagClosure.lean', 66, 'Z')
   ('ExistsLatticeCoordinatesStep.lean', 207, 'Z')
   ('FormsOver.lean', 57, 'C')
   ('FormsOver.lean', 94, 'C')
   ('FormsOver.lean', 101, 'C')
   ('FormsOver.lean', 105, 'C')
   ('GeometrySectionsEqRiemannRoch.lean', 21, '(cxSt F)')
   ('NotLamAreShift.lean', 27, 'Q')
   ('NotLamAreShift.lean', 30, 'Q')
   ('NotLamAreShift.lean', 159, 'Q')
   ('OverIx.lean', 16, '(Q : ∀ D : D7,')
   ('RingKrullDimIntegers.lean', 600, '(C : CxSt F)')
places of the word geoOf: 16 in 10 files
places at which the family Q follows (the term of type GeoP): 4 in 2 files: ['NotLamAreShift.lean', 'OverIx.lean']

NotLamAreShift.lean: 20 declarations; those whose text names geoOf: ['geoOf_set', 'geoOf_surf', 'oAt_geoOf

   geoOf_set    is named at 2 place(s) in 1 file(s) of the archive


   geoOf_surf   is named at 1 place(s) in 1 file(s) of the archive
   oAt_geoOf    is named at 1 place(s) in 1 file(s) of the archive
   places in that file at which an attribute is set on a declaration ('@['): 0



the argument that follows geoCl, at its 163 places:
     151  anyGeometry
      11  Γ
       1  (Γ : GeoP)


anyGeometry: 162 places in 24 files:
     151  in geoCl anyGeometry
       6  the last argument of a lemma
       3  in OX anyGeometry
       1  in OAt anyGeometry
       1  its definition
   the lemmas that take it as their last argument: {'movable_geometric': 2, 'movable_pin': 1, 'slot_mv_extension_with_origin_geo_cl': 2, 'slot_mv_extension_with_list_geo_cl': 1}



argument 1 of OAt, at its 8 places:
       4  (geoCl Γ)
       1  anyGeometry
       1  (geoOf Q)
       1  (Γ : GeoP)
       1  Γ
argument 1 of OX, at its 13 places:
       6  Γ
       3  anyGeometry
       2  p.Γ
       1  (Γ : GeoP)
       1  (geoCl anyGeometry)


argument 3 of parametersOfExtension, at its 291 places:
     240  Γ
      47  (geoCl anyGeometry)
       3  (geoCl Γ)
       1  (Γ : GeoP)

hypotheses_nonempty_of_unramified_exponents_alt, statement:
   places of 'geoCl anyGeometry': 2
   the conditions that the two premises carry: ['FunctionalCondition', 'CovolumeBoundThirdPlaceSeries']
   places of 'parametersOfDependence' (a third premise): 1
   places of 'geoOf': 0
   the statement concludes 'Nonempty (… HypsAlt …)': True
nonempty_hyps_definite_side_of_closed_bound, whole text:
   places of 'geoCl anyGeometry': 3
   places of 'geoOf': 0


## Part 3. The prover's core, version 4.33.1

The three files are fetched from the prover's repository at the tag of the version to which the archive is pinned.

In [14]:
CORE = WORK / "lean_core_4.33.1"
core = {}
for name in ["Init/Prelude.lean", "Init/Core.lean", "Init/Classical.lean"]:
    core[name] = fetch(f"https://raw.githubusercontent.com/leanprover/lean4/{LEAN_TAG}/src/{name}",
                       CORE / name.replace("/", "_"))

P, Co, Cl = core["Init/Prelude.lean"], core["Init/Core.lean"], core["Init/Classical.lean"]
show(P, r"^`False` is the empty proposition", label="Init/Prelude.lean", limit=1)
show(P, r"^inductive False : Prop", label="Init/Prelude.lean")
show(P, r"def Not \(a : Prop\) : Prop", label="Init/Prelude.lean")
show(Co, r"def Ne \{", after=1, label="Init/Core.lean", limit=1)
show(P, r"^axiom Classical\.choice", label="Init/Prelude.lean")
show(Co, r"^axiom propext", label="Init/Core.lean")
show(Co, r"^axiom sound : ", label="Init/Core.lean (in the namespace Quot)")
show(Cl, r"^theorem em \(p : Prop\)", after=4, label="Init/Classical.lean")
show(Cl, r"^theorem propComplete", after=3, label="Init/Classical.lean")
show(Cl, r"propDecidable", label="Init/Classical.lean", limit=2)

Init/Prelude.lean:284: `False` is the empty proposition. Thus, it has no introduction rules.

Init/Prelude.lean:291: inductive False : Prop

Init/Prelude.lean:317: @[implicit_reducible] def Not (a : Prop) : Prop := a → False

Init/Core.lean:875: @[reducible] def Ne {α : Sort u} (a b : α) :=
Init/Core.lean:876:   ¬(a = b)

Init/Prelude.lean:818: axiom Classical.choice {α : Sort u} : Nonempty α → α

Init/Core.lean:1591: axiom propext {a b : Prop} : (a ↔ b) → a = b

Init/Core.lean (in the namespace Quot):1787: axiom sound : ∀ {α : Sort u} {r : α → α → Prop} {a b : α}, r a b → Quot.mk r a = Quot.mk r b

Init/Classical.lean:36: theorem em (p : Prop) : p ∨ ¬p :=
Init/Classical.lean:37:   let U (x : Prop) : Prop := x = True ∨ p
Init/Classical.lean:38:   let V (x : Prop) : Prop := x = False ∨ p
Init/Classical.lean:39:   have exU : ∃ x, U x := ⟨True, Or.inl rfl⟩
Init/Classical.lean:40:   have exV : ∃ x, V x := ⟨False, Or.inl rfl⟩

Init/Classical.lean:128: theorem propComplete (a : Prop) : a = T

## Part 4. The Mathlib library at the pinned commit

The files are fetched from the library's repository at the commit that the archive's manifest names.

In [15]:
ML = WORK / "mathlib_pinned"
PATHS = {
    "RiemannZeta":  "Mathlib/NumberTheory/LSeries/RiemannZeta.lean",
    "HurwitzEven":  "Mathlib/NumberTheory/LSeries/HurwitzZetaEven.lean",
    "ZetaAsymp":    "Mathlib/NumberTheory/Harmonic/ZetaAsymp.lean",
    "Nonvanishing": "Mathlib/NumberTheory/LSeries/Nonvanishing.lean",
    "DirichletCont": "Mathlib/NumberTheory/LSeries/DirichletContinuation.lean",
    "Dirichlet":    "Mathlib/NumberTheory/LSeries/Dirichlet.lean",
    "Gamma":        "Mathlib/Analysis/SpecialFunctions/Gamma/Basic.lean",
    "GammaR":       "Mathlib/Analysis/SpecialFunctions/Gamma/Deligne.lean",
    "GroupWithZero": "Mathlib/Algebra/GroupWithZero/Defs.lean",
    "InfiniteSum":  "Mathlib/Topology/Algebra/InfiniteSum/Defs.lean",
    "PSeries":      "Mathlib/Analysis/PSeriesComplex.lean",
    "NatInt":       "Mathlib/Topology/Algebra/InfiniteSum/NatInt.lean",
    "Bochner":      "Mathlib/MeasureTheory/Integral/Bochner/Basic.lean",
    "Deriv":        "Mathlib/Analysis/Calculus/Deriv/Basic.lean",
}
ml = {k: fetch(f"https://raw.githubusercontent.com/leanprover-community/mathlib4/{MATHLIB_COMMIT}/{p}",
               ML / p.replace("/", "_")) for k, p in PATHS.items()}
for k, p in PATHS.items():
    print(f"{p:62s}{ml[k].stat().st_size:>8,} bytes")

Mathlib/NumberTheory/LSeries/RiemannZeta.lean                   12,306 bytes
Mathlib/NumberTheory/LSeries/HurwitzZetaEven.lean               39,471 bytes
Mathlib/NumberTheory/Harmonic/ZetaAsymp.lean                    32,972 bytes
Mathlib/NumberTheory/LSeries/Nonvanishing.lean                  21,780 bytes
Mathlib/NumberTheory/LSeries/DirichletContinuation.lean         18,979 bytes
Mathlib/NumberTheory/LSeries/Dirichlet.lean                     19,967 bytes
Mathlib/Analysis/SpecialFunctions/Gamma/Basic.lean              22,917 bytes
Mathlib/Analysis/SpecialFunctions/Gamma/Deligne.lean             9,148 bytes
Mathlib/Algebra/GroupWithZero/Defs.lean                         14,279 bytes
Mathlib/Topology/Algebra/InfiniteSum/Defs.lean                  16,486 bytes
Mathlib/Analysis/PSeriesComplex.lean                             1,031 bytes
Mathlib/Topology/Algebra/InfiniteSum/NatInt.lean                26,028 bytes
Mathlib/MeasureTheory/Integral/Bochner/Basic.lean               70,432 bytes

### The zeta function and the hypothesis

In [16]:
L = PATHS
show(ml["RiemannZeta"], r"^def riemannZeta\b", label=L["RiemannZeta"])
show(ml["HurwitzEven"], r"^noncomputable def hurwitzZetaEven\b", after=2, label=L["HurwitzEven"])
show(ml["RiemannZeta"], r"^def RiemannHypothesis\b", after=1, label=L["RiemannZeta"])
show(ml["RiemannZeta"], r"^theorem zeta_eq_tsum_one_div_nat_add_one_cpow\b", after=1, label=L["RiemannZeta"])
show(ml["RiemannZeta"], r"^theorem differentiableAt_riemannZeta\b", label=L["RiemannZeta"])
show(ml["RiemannZeta"], r"^theorem riemannZeta_one_sub\b", after=1, label=L["RiemannZeta"])
show(ml["Nonvanishing"], r"^lemma _root_\.riemannZeta_ne_zero_of_one_le_re\b", after=1, label=L["Nonvanishing"])
show(ml["DirichletCont"], r"lemma LFunction_modOne_eq\b", after=1, label=L["DirichletCont"])

Mathlib/NumberTheory/LSeries/RiemannZeta.lean:121: def riemannZeta := hurwitzZetaEven 0

Mathlib/NumberTheory/LSeries/HurwitzZetaEven.lean:583: noncomputable def hurwitzZetaEven (a : UnitAddCircle) :=
Mathlib/NumberTheory/LSeries/HurwitzZetaEven.lean:584:   Function.update (fun s ↦ completedHurwitzZetaEven a s / Gammaℝ s)
Mathlib/NumberTheory/LSeries/HurwitzZetaEven.lean:585:   0 (if a = 0 then -1 / 2 else 0)

Mathlib/NumberTheory/LSeries/RiemannZeta.lean:185: def RiemannHypothesis : Prop :=
Mathlib/NumberTheory/LSeries/RiemannZeta.lean:186:   ∀ (s : ℂ) (_ : riemannZeta s = 0) (_ : ¬∃ n : ℕ, s = -2 * (n + 1)) (_ : s ≠ 1), s.re = 1 / 2

Mathlib/NumberTheory/LSeries/RiemannZeta.lean:214: theorem zeta_eq_tsum_one_div_nat_add_one_cpow {s : ℂ} (hs : 1 < re s) :
Mathlib/NumberTheory/LSeries/RiemannZeta.lean:215:     riemannZeta s = ∑' n : ℕ, 1 / (n + 1 : ℂ) ^ s := by

Mathlib/NumberTheory/LSeries/RiemannZeta.lean:139: theorem differentiableAt_riemannZeta {s : ℂ} (hs' : s ≠ 1) : Differentiabl

### The clauses that write a value where a rule withholds one

In [17]:
print("— the inverse of \"0\", and the guarded clause for inverses —")
show(ml["GroupWithZero"], r"^class GroupWithZero ", label=L["GroupWithZero"])
show(ml["GroupWithZero"], r"protected inv_zero", before=1, label=L["GroupWithZero"])
show(ml["GroupWithZero"], r"protected mul_inv_cancel", before=1, label=L["GroupWithZero"])
print("  (the two fields are GroupWithZero.inv_zero and GroupWithZero.mul_inv_cancel)")
print("— the sum of a family that has no sum —")
show(ml["InfiniteSum"], r"is the unconditional sum of `f` if it exists, or 0 otherwise", label=L["InfiniteSum"])
print("  (the library states the next two facts for products; the attribute to_additive generates")
print("   from them the forms for sums, tsum_eq_zero_of_not_summable and summable_nat_add_iff)")
show(ml["InfiniteSum"], r"^theorem tprod_eq_one_of_not_multipliable\b", before=1, label=L["InfiniteSum"])
show(ml["NatInt"], r"^theorem multipliable_nat_add_iff\b", before=1, after=1, label=L["NatInt"])
print("— the series converges exactly on the half-plane —")
show(ml["PSeries"], r"^lemma Complex\.summable_one_div_nat_cpow\b", after=1, label=L["PSeries"])
print("— the gamma function at its poles —")
show(ml["Gamma"], r"^theorem Gamma_neg_nat_eq_zero\b", before=1, label=L["Gamma"], limit=1)
print("  (the theorem stands in the namespace Complex; the file refers to it by its full name:)")
show(ml["Gamma"], r"Complex\.Gamma_neg_nat_eq_zero", label=L["Gamma"], limit=1)
show(ml["GammaR"], r"^lemma Gammaℝ_eq_zero_iff\b", label=L["GammaR"])
print("— the zeta function at the point 0 —")
show(ml["RiemannZeta"], r"^theorem riemannZeta_zero\b", label=L["RiemannZeta"])
print("— the trivial balance points —")
show(ml["RiemannZeta"], r"^theorem riemannZeta_neg_two_mul_nat_add_one\b", after=1, label=L["RiemannZeta"])
show(ml["HurwitzEven"], r"^theorem hurwitzZetaEven_neg_two_mul_nat_add_one\b", after=5, label=L["HurwitzEven"])
print("— the zeta function at its pole —")
show(ml["ZetaAsymp"], r"^lemma riemannZeta_one\b", before=2, label=L["ZetaAsymp"])
show(ml["ZetaAsymp"], r"^lemma riemannZeta_one_ne_zero\b", before=1, label=L["ZetaAsymp"])

— the inverse of "0", and the guarded clause for inverses —
Mathlib/Algebra/GroupWithZero/Defs.lean:227: class GroupWithZero (G₀ : Type u) extends MonoidWithZero G₀, DivInvMonoid G₀, Nontrivial G₀ where

Mathlib/Algebra/GroupWithZero/Defs.lean:228:   /-- The inverse of `0` in a group with zero is `0`. -/
Mathlib/Algebra/GroupWithZero/Defs.lean:229:   protected inv_zero : (0 : G₀)⁻¹ = 0

Mathlib/Algebra/GroupWithZero/Defs.lean:230:   /-- Every nonzero element of a group with zero is invertible. -/
Mathlib/Algebra/GroupWithZero/Defs.lean:231:   protected mul_inv_cancel (a : G₀) : a ≠ 0 → a * a⁻¹ = 1

  (the two fields are GroupWithZero.inv_zero and GroupWithZero.mul_inv_cancel)
— the sum of a family that has no sum —
Mathlib/Topology/Algebra/InfiniteSum/Defs.lean:132: @[to_additive /-- `∑' i, f i` is the unconditional sum of `f` if it exists, or 0 otherwise.

  (the library states the next two facts for products; the attribute to_additive generates
   from them the forms for sums, tsum_e

### The clauses for the integral and for the derivative, and the lemmas that read them backwards

The library defines the integral of every function. Where the function is integrable the definition issues the integral; elsewhere it issues "0". The theorem `integral_undef` states the clause, and `Integrable.of_integral_ne_zero` concludes integrability from an integral apart from "0"; its proof is one application of `Not.imp_symm` to the clause. The derivative has the same pair: `deriv_zero_of_not_differentiableAt` states the clause, and `differentiableAt_of_deriv_ne_zero` reads it backwards.

In [18]:
show(ml["Bochner"], r"^irreducible_def integral\b", before=1, after=3, label=L["Bochner"])
show(ml["Bochner"], r"^theorem integral_undef\b", label=L["Bochner"])
show(ml["Bochner"], r"^theorem Integrable\.of_integral_ne_zero\b", after=1, label=L["Bochner"])
print("— the derivative of a function at a point where it has none —")
show(ml["Deriv"], r"^theorem deriv_zero_of_not_differentiableAt\b", label=L["Deriv"])
show(ml["Deriv"], r"^theorem differentiableAt_of_deriv_ne_zero\b", after=1, label=L["Deriv"])

Mathlib/MeasureTheory/Integral/Bochner/Basic.lean:155: /-- The Bochner integral -/
Mathlib/MeasureTheory/Integral/Bochner/Basic.lean:156: irreducible_def integral {_ : MeasurableSpace α} (μ : Measure α) (f : α → G) : G :=
Mathlib/MeasureTheory/Integral/Bochner/Basic.lean:157:   if _ : CompleteSpace G then
Mathlib/MeasureTheory/Integral/Bochner/Basic.lean:158:     if hf : Integrable f μ then L1.integral (hf.toL1 f) else 0
Mathlib/MeasureTheory/Integral/Bochner/Basic.lean:159:   else 0

Mathlib/MeasureTheory/Integral/Bochner/Basic.lean:202: theorem integral_undef {f : α → G} (h : ¬Integrable f μ) : ∫ a, f a ∂μ = 0 := by

Mathlib/MeasureTheory/Integral/Bochner/Basic.lean:206: theorem Integrable.of_integral_ne_zero {f : α → G} (h : ∫ a, f a ∂μ ≠ 0) : Integrable f μ :=
Mathlib/MeasureTheory/Integral/Bochner/Basic.lean:207:   Not.imp_symm integral_undef h

— the derivative of a function at a point where it has none —
Mathlib/Analysis/Calculus/Deriv/Basic.lean:250: theorem deriv_zero_of_not_d

### The identities on the half-plane where the series converge

Each of the first four statements carries the hypothesis that the real part of $s$ exceeds $1$. In the library's notation `L` is the series of a sequence, `↗χ` the sequence of the values of a character, and `Λ` the function of von Mangoldt. The last three statements are about the continued functions: the $L$-function of a character apart from the trivial one is differentiable at every point of the plane, an $L$-function is the $L$-function of its primitive character times a finite product over primes, and an $L$-function is apart from "0" where the real part of $s$ is at least $1$.

In [19]:
show(ml["Dirichlet"], r"^lemma LSeries_twist_vonMangoldt_eq\b", after=1, label=L["Dirichlet"])
show(ml["Dirichlet"], r"^lemma LSeries_vonMangoldt_eq_deriv_riemannZeta_div\b", after=1, label=L["Dirichlet"])
show(ml["DirichletCont"], r"^lemma LFunction_eq_LSeries\b", after=1, label=L["DirichletCont"])
show(ml["DirichletCont"], r"^lemma deriv_LFunction_eq_deriv_LSeries\b", after=1, label=L["DirichletCont"])
show(ml["DirichletCont"], r"^lemma differentiable_LFunction\b", after=1, label=L["DirichletCont"])
show(ml["DirichletCont"], r"^lemma LFunction_changeLevel\b", after=4, label=L["DirichletCont"])
show(ml["Nonvanishing"], r"^theorem LFunction_ne_zero_of_one_le_re\b", after=1, label=L["Nonvanishing"])

Mathlib/NumberTheory/LSeries/Dirichlet.lean:409: lemma LSeries_twist_vonMangoldt_eq {N : ℕ} (χ : DirichletCharacter ℂ N) {s : ℂ} (hs : 1 < s.re) :
Mathlib/NumberTheory/LSeries/Dirichlet.lean:410:     L (↗χ * ↗Λ) s = -deriv (L ↗χ) s / L ↗χ s := by

Mathlib/NumberTheory/LSeries/Dirichlet.lean:436: lemma LSeries_vonMangoldt_eq_deriv_riemannZeta_div {s : ℂ} (hs : 1 < s.re) :
Mathlib/NumberTheory/LSeries/Dirichlet.lean:437:     L ↗Λ s = - deriv riemannZeta s / riemannZeta s := by

Mathlib/NumberTheory/LSeries/DirichletContinuation.lean:75: lemma LFunction_eq_LSeries (χ : DirichletCharacter ℂ N) {s : ℂ} (hs : 1 < re s) :
Mathlib/NumberTheory/LSeries/DirichletContinuation.lean:76:     LFunction χ s = LSeries (χ ·) s :=

Mathlib/NumberTheory/LSeries/DirichletContinuation.lean:79: lemma deriv_LFunction_eq_deriv_LSeries (χ : DirichletCharacter ℂ N) {s : ℂ} (hs : 1 < s.re) :
Mathlib/NumberTheory/LSeries/DirichletContinuation.lean:80:     deriv (LFunction χ) s = deriv (LSeries (χ ·)) s := by

Math

## Part 5. The note's own statements, checked by the prover

The cell writes the Lean file of the note, compiles it with the prover, and prints the prover's output: for each declaration, the axioms on which it rests, and the two results of running the search. The file imports nothing beyond the prover's prelude. The prover is taken from the command `lean` on the path; its version is printed.

In [20]:
LEAN_SOURCE = r'''/-
Alpöge's Certified Lean Derivation Does Not Prove the Riemann Hypothesis:
Granting Every Check, the Hypothesis Follows from a Premise Stronger Than Itself

Parker Emmerson (Yaohushuason), November 2026.

The statements of the note that concern the prover's own rules, checked in the core of
Lean 4.33.1, the version to which the certified archive is pinned. The file imports
nothing beyond the prover's prelude. Each declaration names the statement of the note
that it checks; the last lines print the axioms on which each declaration rests.
-/

namespace CompletenessOfValues

universe u

/-! ## Terms completed by clause in the prover's core
    (the proposition "The clauses, and the term of the certified sentence") -/

/-- Division of natural numbers is completed at the divisor "0" by the value "0". -/
theorem nat_div_completed (n : Nat) : n / 0 = 0 := Nat.div_zero n

/-- The remainder is completed at the divisor "0" by the dividend. -/
theorem nat_mod_completed (n : Nat) : n % 0 = n := Nat.mod_zero n

/-- Subtraction of natural numbers is completed, where the subtrahend is the larger, by "0". -/
theorem nat_sub_completed : (2 : Nat) - 5 = 0 := rfl

/-- The predecessor is completed at "0" by "0". -/
theorem nat_pred_completed : Nat.pred 0 = 0 := rfl

/-! ## Statements: `False`, the counter-inscription, and "apart from" -/

/-- The counter-inscription of a statement is, by definition, "the statement yields `False`". -/
theorem counter_inscription_unfolds (a : Prop) : (¬ a) = (a → False) := rfl

/-- "Apart from" is, by definition, "equality yields `False`". -/
theorem apart_unfolds {α : Sort u} (x y : α) : (x ≠ y) = (x = y → False) := rfl

/-- `False` has no rule of derivation: a case analysis on a derivation of it has no case,
and so yields every statement. -/
theorem false_has_no_case (h : False) (q : Prop) : q := nomatch h

/-! ## Every statement is decided in value
    (the theorem "Every statement is decided in value") -/

/-- Part (1): the prover's own theorem `Classical.propComplete`. -/
theorem decided_in_value (a : Prop) : a = True ∨ a = False := Classical.propComplete a

/-- Part (2): the excluded middle derived from the selection axiom, by the argument of
Diaconescu (1975), written step by step. The prover's core library proves the same statement
by the same argument (`Classical.em`). -/
theorem excluded_middle_from_selection (p : Prop) : p ∨ ¬ p := by
  -- Two collections of statements: `U` holds of `True`, `V` holds of `False`, and both hold
  -- of every statement when `p` holds.
  let U : Prop → Prop := fun x => x = True ∨ p
  let V : Prop → Prop := fun x => x = False ∨ p
  have exU : ∃ x, U x := ⟨True, Or.inl rfl⟩
  have exV : ∃ x, V x := ⟨False, Or.inl rfl⟩
  -- The selection axiom issues a member of each collection.
  have hu : U (Classical.choose exU) := Classical.choose_spec exU
  have hv : V (Classical.choose exV) := Classical.choose_spec exV
  -- If `p` holds, the two collections are one collection, by function extensionality and
  -- propositional extensionality, and the selection issues one member for both.
  have same : p → Classical.choose exU = Classical.choose exV := by
    intro hp
    have hUV : U = V :=
      funext fun x => propext ⟨fun _ => Or.inr hp, fun _ => Or.inr hp⟩
    have key : ∀ (W : Prop → Prop) (e : U = W) (ex : ∃ x, W x),
        Classical.choose exU = Classical.choose ex := by
      intro W e
      subst e
      intro ex
      rfl
    exact key V hUV exV
  -- The cases.
  cases hu with
  | inr hp => exact Or.inl hp
  | inl huT =>
    cases hv with
    | inr hp => exact Or.inl hp
    | inl hvF =>
      refine Or.inr (fun hp => ?_)
      have e : True = False := huT.symm.trans ((same hp).trans hvF)
      exact e.mp True.intro

/-- From the excluded middle and propositional extensionality, the value of every statement. -/
theorem decided_in_value_from_selection (a : Prop) : a = True ∨ a = False :=
  match excluded_middle_from_selection a with
  | Or.inl h => Or.inl (propext ⟨fun _ => True.intro, fun _ => h⟩)
  | Or.inr h => Or.inr (propext ⟨fun ha => h ha, fun hf => nomatch hf⟩)

/-- Part (3): the rules define a function from the statements to the two values. The prover
requires the mark `noncomputable`: the function is defined and no program computes it. -/
noncomputable def value (a : Prop) : Bool := @decide a (Classical.propDecidable a)

theorem value_true_iff (a : Prop) : value a = true ↔ a := by
  unfold value
  exact @decide_eq_true_iff a (Classical.propDecidable a)

theorem value_false_iff (a : Prop) : value a = false ↔ ¬ a := by
  unfold value
  exact @decide_eq_false_iff_not a (Classical.propDecidable a)

/-- Every statement receives one of the two values. -/
theorem value_total (a : Prop) : value a = true ∨ value a = false :=
  match value a with
  | true => Or.inl rfl
  | false => Or.inr rfl

/-! ## Formation data, completions, and the raw sentence
    (the section "The certified derivation is not a derivation from the zeta series") -/

/-- The shape of the classical sentence: positions, values, the region on which a clause
presents values, a strip, a line, the clause, and the constant of the matrix. -/
structure Formation where
  Pos : Type
  Val : Type
  presented : Pos → Prop
  strip : Pos → Prop
  line : Pos → Prop
  clause : (s : Pos) → presented s → Val
  z : Val

/-- A completion agrees with the clause wherever the clause presents a value. -/
def IsCompletion (F : Formation) (f : F.Pos → F.Val) : Prop :=
  ∀ (s : F.Pos) (h : F.presented s), f s = F.clause s h

/-- The raw sentence at a function: every position of the strip at which the function issues
the constant of the matrix lies on the line. -/
def Raw (F : Formation) (f : F.Pos → F.Val) : Prop :=
  ∀ s : F.Pos, F.strip s → f s = F.z → F.line s

/-- The three facts: the clause withholds its value on the strip; the strip has a position off
the line; and a value apart from the constant of the matrix exists. -/
structure Facts (F : Formation) : Prop where
  strip_withheld : ∀ s, F.strip s → ¬ F.presented s
  off_line : ∃ p, F.strip p ∧ ¬ F.line p
  other_value : ∃ u : F.Val, u ≠ F.z

/-- The completion that issues `w` wherever the clause withholds a value. -/
noncomputable def fill (F : Formation) (w : F.Pos → F.Val) : F.Pos → F.Val :=
  fun s => @dite F.Val (F.presented s) (Classical.propDecidable _) (fun h => F.clause s h) (fun _ => w s)

theorem fill_isCompletion (F : Formation) (w : F.Pos → F.Val) : IsCompletion F (fill F w) := by
  intro s h
  unfold fill
  exact @dif_pos (F.presented s) (Classical.propDecidable _) h F.Val _ _

theorem fill_off (F : Formation) (w : F.Pos → F.Val) (s : F.Pos) (h : ¬ F.presented s) :
    fill F w s = w s := by
  unfold fill
  exact @dif_neg (F.presented s) (Classical.propDecidable _) h F.Val _ _

/-- The theorem "The formation data give the sentence both truth values", part (1): one
completion makes the raw sentence true and another makes it false. -/
theorem both_values {F : Formation} (hF : Facts F) :
    (∃ f, IsCompletion F f ∧ Raw F f) ∧ (∃ g, IsCompletion F g ∧ ¬ Raw F g) := by
  obtain ⟨u, hu⟩ := hF.other_value
  obtain ⟨p, hp, hpl⟩ := hF.off_line
  refine ⟨⟨fill F (fun _ => u), fill_isCompletion F _, ?_⟩,
          ⟨fill F (fun _ => F.z), fill_isCompletion F _, ?_⟩⟩
  · intro s hs hz
    have e : fill F (fun _ => u) s = u := fill_off F _ s (hF.strip_withheld s hs)
    exact absurd (e.symm.trans hz) hu
  · intro h
    have e : fill F (fun _ => F.z) p = F.z := fill_off F _ p (hF.strip_withheld p hp)
    exact hpl (h p hp e)

/-- A system of rules derives properties of functions; it is sound for the completions when
whatever it derives holds at every completion. -/
structure System (F : Formation) where
  derives : ((F.Pos → F.Val) → Prop) → Prop

def System.Sound {F : Formation} (S : System F) : Prop :=
  ∀ φ, S.derives φ → ∀ f, IsCompletion F f → φ f

/-- The same theorem, part (2): every system of rules sound for the completions leaves the raw
sentence and its counter-inscription both underived. -/
theorem underived_from_formation_data {F : Formation} (hF : Facts F) (S : System F)
    (hS : S.Sound) : ¬ S.derives (Raw F) ∧ ¬ S.derives (fun f => ¬ Raw F f) := by
  obtain ⟨⟨f, hf, hfr⟩, ⟨g, hg, hgr⟩⟩ := both_values hF
  exact ⟨fun h => hgr (hS _ h g hg), fun h => hS _ h f hf hfr⟩

/-- The theorem "The analytic selection", the abstract part: when an admissibility clause leaves
one completion, every admissible completion gives the raw sentence the same truth value. -/
theorem selection_fixes {F : Formation} (adm : (F.Pos → F.Val) → Prop)
    (one : ∀ f g, IsCompletion F f → IsCompletion F g → adm f → adm g → f = g)
    (f g : F.Pos → F.Val) (hf : IsCompletion F f) (hg : IsCompletion F g)
    (af : adm f) (ag : adm g) : Raw F f ↔ Raw F g := by
  rw [one f g hf hg af ag]

/-- The corollary "The value before the derivation": at every function the raw sentence is
decided in value, by one application of `decided_in_value`; the derivation supplies neither of
the two values. -/
theorem selected_sentence_has_value {F : Formation} (f : F.Pos → F.Val) :
    Raw F f = True ∨ Raw F f = False :=
  decided_in_value (Raw F f)

/-! ## Refutation, presentation, and Markov's principle
    (the theorem "Refutation, presented bound, and Markov's principle") -/

/-- Markov's principle: a sequence of decisions that is refuted to fail at every place
succeeds at some place. -/
def MarkovPrinciple : Prop :=
  ∀ d : Nat → Bool, ¬ (∀ n, d n = false) → ∃ n, d n = true

/-- The certificate passage: at every window, a search whose failure at every candidate is
refuted finds a certificate. -/
def CertificatePassage : Prop :=
  ∀ cert : Nat → Nat → Bool, (∀ w, ¬ ∀ c, cert w c = false) → ∀ w, ∃ c, cert w c = true

/-- The certificate passage is Markov's principle. -/
theorem certificate_passage_iff_markov : CertificatePassage ↔ MarkovPrinciple :=
  ⟨fun hP d hd => hP (fun _ c => d c) (fun _ => hd) 0, fun hM cert h w => hM (cert w) (h w)⟩

/-- In the completed rules Markov's principle is a theorem: the excluded middle supplies it. -/
theorem markov_in_completed_rules : MarkovPrinciple := by
  intro d h
  apply Classical.byContradiction
  intro hne
  apply h
  intro n
  cases hd : d n with
  | false => rfl
  | true => exact absurd ⟨n, hd⟩ hne

/-- `Halts d n`: the search that tests `d` at `n`, `n + 1`, and so on, stops. Its one rule:
the search stops from `n` when, if `d` fails at `n`, the search stops from `n + 1`. -/
inductive Halts (d : Nat → Bool) : Nat → Prop
  | step (n : Nat) (h : d n = false → Halts d (n + 1)) : Halts d n

theorem Halts.next {d : Nat → Bool} {n : Nat} (h : Halts d n) (hd : d n = false) :
    Halts d (n + 1) :=
  match h with
  | .step _ f => f hd

theorem halts_before (d : Nat → Bool) (m : Nat) (hm : d m = true) :
    ∀ k n, n + k = m → Halts d n := by
  intro k
  induction k with
  | zero =>
    intro n hn
    have e : n = m := hn
    subst e
    exact Halts.step n (fun hf => Bool.noConfusion (hm.symm.trans hf))
  | succ k ih =>
    intro n hn
    exact Halts.step n
      (fun _ => ih (n + 1) (by rw [Nat.add_assoc, Nat.add_comm 1 k]; exact hn))

/-- An existence statement yields that the search from the first place stops. -/
theorem halts_of_exists (d : Nat → Bool) (h : ∃ n, d n = true) : Halts d 0 :=
  match h with
  | ⟨m, hm⟩ => halts_before d m hm m 0 (Nat.zero_add m)

/-- One step of the search, as a relation on the places from which the search stops. -/
def Step (d : Nat → Bool) (a b : { n : Nat // Halts d n }) : Prop :=
  a.1 = b.1 + 1 ∧ d b.1 = false

theorem step_wf (d : Nat → Bool) : WellFounded (Step d) := by
  constructor
  intro a
  obtain ⟨n, h⟩ := a
  induction h with
  | step n f ih =>
    constructor
    intro b hb
    obtain ⟨m, hm⟩ := b
    obtain ⟨e, hf⟩ := hb
    have e' : m = n + 1 := e
    subst e'
    exact ih hf

instance (d : Nat → Bool) : WellFoundedRelation { n : Nat // Halts d n } where
  rel := Step d
  wf := step_wf d

/-- The search: test `d` at the place; stop there if it succeeds; otherwise go on. -/
def searchFrom (d : Nat → Bool) (a : { n : Nat // Halts d n }) : { m : Nat // d m = true } :=
  match hd : d a.1 with
  | true => ⟨a.1, hd⟩
  | false => searchFrom d ⟨a.1 + 1, a.2.next hd⟩
termination_by a
decreasing_by exact ⟨rfl, hd⟩

/-- From an existence statement to a presented number, by the search. No axiom is used. -/
def present (d : Nat → Bool) (h : ∃ n, d n = true) : { m : Nat // d m = true } :=
  searchFrom d ⟨0, halts_of_exists d h⟩

/-- From a refutation to a presented number: Markov's principle, then the search. -/
def presentFromRefutation (d : Nat → Bool) (h : ¬ ∀ n, d n = false) :
    { m : Nat // d m = true } :=
  present d (markov_in_completed_rules d h)

/-- A demonstration: the decision that succeeds at the number 7 and at no other. -/
def demo : Nat → Bool := fun n => n == 7

theorem demo_refuted : ¬ ∀ n, demo n = false :=
  fun h => Bool.noConfusion ((h 7).symm.trans (rfl : demo 7 = true))

/-! ## A clause read backwards, and the excluded middle
    (the theorem "The backward reading of a clause is the excluded middle") -/

/-- An operation completed by a clause. Where the rule presents a value (the statement `P`
holds, and the value is `rule h`), the operation issues that value; elsewhere it issues the
default `d`. The library's integral has this shape, with `P` the statement that the function
is integrable and `d` the value "0". The prover requires the mark `noncomputable`. -/
noncomputable def completed {α : Type} (P : Prop) (rule : P → α) (d : α) : α :=
  @dite α P (Classical.propDecidable P) (fun h => rule h) (fun _ => d)

/-- Where the rule presents a value, the completed operation issues it. -/
theorem completed_presented {α : Type} (P : Prop) (rule : P → α) (d : α) (h : P) :
    completed P rule d = rule h := by
  unfold completed
  exact @dif_pos P (Classical.propDecidable P) h α _ _

/-- The clause: where the rule withholds a value, the completed operation issues the default. -/
theorem completed_clause {α : Type} (P : Prop) (rule : P → α) (d : α) (h : ¬ P) :
    completed P rule d = d := by
  unfold completed
  exact @dif_neg P (Classical.propDecidable P) h α _ _

/-- The clause read backwards, in the completed rules: if the completed operation issues a
value apart from the default, then the rule presents a value. This is the inference of the
library's lemma `Integrable.of_integral_ne_zero`. -/
theorem presented_of_apart {α : Type} (P : Prop) (rule : P → α) (d : α)
    (h : completed P rule d ≠ d) : P :=
  Classical.byContradiction (fun hn => h (completed_clause P rule d hn))

/-- Without the excluded middle, a clause and a value apart from the default give the
refutation of the refutation of `P`, and no more. -/
theorem clause_gives_double_negation {α : Type} {P : Prop} {v d : α}
    (clause : ¬ P → v = d) (apart : v ≠ d) : ¬ ¬ P :=
  fun hn => apart (clause hn)

/-- The backward reading of a clause, as a schema over every type, every statement and every
pair of values: from "if `P` is refuted then `v = d`" and "`v` is apart from `d`", conclude `P`. -/
def BackwardReading : Prop :=
  ∀ (α : Type) (P : Prop) (v d : α), (¬ P → v = d) → v ≠ d → P

/-- The excluded middle, as a schema over every statement. -/
def ExcludedMiddle : Prop := ∀ p : Prop, p ∨ ¬ p

/-- The excluded middle yields the backward reading. -/
theorem backward_of_excluded_middle (em : ExcludedMiddle) : BackwardReading :=
  fun _ P _ _ clause apart =>
    match em P with
    | Or.inl h => h
    | Or.inr hn => absurd (clause hn) apart

/-- The backward reading, taken at the type of statements with the default `False`, removes a
double counter-inscription: the clause "if `q` is refuted then `q = False`" holds by
propositional extensionality, and `q` is apart from `False` when its refutation is refuted. -/
theorem double_negation_of_backward (B : BackwardReading) (q : Prop) (hnn : ¬ ¬ q) : q :=
  B Prop q q False
    (fun hn => propext ⟨fun hq => absurd hq hn, fun hf => False.elim hf⟩)
    (fun heq => hnn (fun hq => Eq.mp heq hq))

/-- The backward reading yields the excluded middle. -/
theorem excluded_middle_of_backward (B : BackwardReading) : ExcludedMiddle :=
  fun p => double_negation_of_backward B (p ∨ ¬ p)
    (fun h => h (Or.inr (fun hp => h (Or.inl hp))))

/-- The backward reading of a clause is the excluded middle. -/
theorem backward_reading_iff_excluded_middle : BackwardReading ↔ ExcludedMiddle :=
  ⟨excluded_middle_of_backward, backward_of_excluded_middle⟩

/-- In the completed rules the backward reading holds. -/
theorem backward_in_completed_rules : BackwardReading :=
  backward_of_excluded_middle Classical.em

/-! ## The sponsor of a derived law

`L` stands for "the law holds at every whole number" and `K` for "the rules are consistent."
What a certified derivation of the law establishes outright is the conditional `¬ L → ¬ K`:
a whole number at which the law failed would yield a clash. -/

/-- Proposition "A derived law over the whole numbers", the step: if each instance at which
`Q` held would yield a clash, then on the sponsor `K` the law holds. -/
theorem derived_law_on_sponsor {Q : Nat → Prop} {K : Prop}
    (clash : ∀ n, Q n → ¬ K) (hK : K) : ∀ n, ¬ Q n :=
  fun n hq => clash n hq hK

/-- The conditional excludes the fourth combination: the law fails and the rules are
consistent. -/
theorem excludes_fourth_case {L K : Prop} (h : ¬ L → ¬ K) : ¬ (¬ L ∧ K) :=
  fun hLK => h hLK.1 hLK.2

/-- The conditional holds in each of the first three combinations, so it excludes none of
them: the law and consistency; the law and a clash; no law and a clash. -/
theorem first_three_cases_admitted :
    (¬ True → ¬ True) ∧ (¬ True → ¬ False) ∧ (¬ False → ¬ False) :=
  ⟨fun h _ => h trivial, fun _ h => h, fun _ h => h⟩

/-- The conditional fails in the fourth combination. -/
theorem fourth_case_refuted : ¬ (¬ False → ¬ True) :=
  fun h => h (fun f => f) trivial

/-- The conditional is not the statement that exactly one of "the law" and "a clash" holds:
it admits the combination in which both hold. -/
theorem conditional_is_not_exclusive :
    ¬ (∀ L K : Prop, (¬ L → ¬ K) → ¬ (L ∧ ¬ K)) :=
  fun h => h True False (fun _ f => f) ⟨trivial, fun f => f⟩

/-- The step of the progression of sponsors: if no system of the progression derives its own
consistency, then none derives the consistency of all of them. Here `derives N n` reads "the
`N`-th system derives the consistency of the `n`-th." -/
theorem no_rung_closes (derives : Nat → Nat → Prop) (own : ∀ n, ¬ derives n n) :
    ¬ ∃ N, ∀ n, derives N n :=
  fun ⟨N, h⟩ => own N (h N)

/-! ## Löb's theorem, from the derivability conditions and one fixed point -/

/-- A system of derivation: formulas with a conditional `imp` and a formula `box a` read
"`a` is derived"; a relation `D` read "is derived"; the rule of detachment; the two schemes
of the conditional; and the three derivability conditions. -/
structure Derivability where
  Form : Type
  imp : Form → Form → Form
  box : Form → Form
  D : Form → Prop
  mp : ∀ {a b : Form}, D (imp a b) → D a → D b
  ax_k : ∀ a b : Form, D (imp a (imp b a))
  ax_s : ∀ a b c : Form, D (imp (imp a (imp b c)) (imp (imp a b) (imp a c)))
  d1 : ∀ {a : Form}, D a → D (box a)
  d2 : ∀ a b : Form, D (imp (box (imp a b)) (imp (box a) (box b)))
  d3 : ∀ a : Form, D (imp (box a) (box (box a)))

namespace Derivability

variable (S : Derivability)

/-- Chaining two derived conditionals. -/
theorem chain {a b c : S.Form} (h1 : S.D (S.imp a b)) (h2 : S.D (S.imp b c)) :
    S.D (S.imp a c) :=
  S.mp (S.mp (S.ax_s a b c) (S.mp (S.ax_k (S.imp b c) a) h2)) h1

/-- From `a → (b → c)` and `a → b`, the conditional `a → c`. -/
theorem under {a b c : S.Form} (h1 : S.D (S.imp a (S.imp b c))) (h2 : S.D (S.imp a b)) :
    S.D (S.imp a c) :=
  S.mp (S.mp (S.ax_s a b c) h1) h2

/-- Theorem "Löb, 1955", the passage from the conditions and the fixed point: if
`box a → a` is derived, and `psi` is a fixed point of "if `box x` then `a`", then `a` is
derived. -/
theorem lob {a psi : S.Form}
    (fix1 : S.D (S.imp psi (S.imp (S.box psi) a)))
    (fix2 : S.D (S.imp (S.imp (S.box psi) a) psi))
    (h : S.D (S.imp (S.box a) a)) : S.D a :=
  have s1 : S.D (S.imp (S.box psi) (S.box (S.imp (S.box psi) a))) :=
    S.mp (S.d2 psi (S.imp (S.box psi) a)) (S.d1 fix1)
  have s2 : S.D (S.imp (S.box psi) (S.imp (S.box (S.box psi)) (S.box a))) :=
    S.chain s1 (S.d2 (S.box psi) a)
  have s3 : S.D (S.imp (S.box psi) (S.box a)) := S.under s2 (S.d3 psi)
  have s4 : S.D (S.imp (S.box psi) a) := S.chain s3 h
  have s5 : S.D psi := S.mp fix2 s4
  S.mp s4 (S.d1 s5)

/-- With the fixed point, `box a → a` is derived exactly when `a` is. -/
theorem lob_iff {a psi : S.Form}
    (fix1 : S.D (S.imp psi (S.imp (S.box psi) a)))
    (fix2 : S.D (S.imp (S.imp (S.box psi) a) psi)) :
    S.D (S.imp (S.box a) a) ↔ S.D a :=
  ⟨S.lob fix1 fix2, fun ha => S.mp (S.ax_k a (S.box a)) ha⟩

/-- The second incompleteness theorem as an instance: with `bot` for a clash, the formula
`box bot → bot` says that the system derives no clash, and a system that derives no clash
does not derive it. -/
theorem underived_consistency {bot psi : S.Form}
    (fix1 : S.D (S.imp psi (S.imp (S.box psi) bot)))
    (fix2 : S.D (S.imp (S.imp (S.box psi) bot) psi))
    (consistent : ¬ S.D bot) : ¬ S.D (S.imp (S.box bot) bot) :=
  fun h => consistent (S.lob fix1 fix2 h)

end Derivability

end CompletenessOfValues

open CompletenessOfValues

/-! ## The axiom report -/

#print axioms nat_div_completed
#print axioms nat_mod_completed
#print axioms nat_sub_completed
#print axioms nat_pred_completed
#print axioms counter_inscription_unfolds
#print axioms apart_unfolds
#print axioms false_has_no_case
#print axioms decided_in_value
#print axioms excluded_middle_from_selection
#print axioms decided_in_value_from_selection
#print axioms value
#print axioms value_true_iff
#print axioms value_false_iff
#print axioms value_total
#print axioms both_values
#print axioms underived_from_formation_data
#print axioms selection_fixes
#print axioms selected_sentence_has_value
#print axioms certificate_passage_iff_markov
#print axioms markov_in_completed_rules
#print axioms halts_of_exists
#print axioms step_wf
#print axioms searchFrom
#print axioms present
#print axioms presentFromRefutation
#print axioms completed
#print axioms completed_presented
#print axioms completed_clause
#print axioms presented_of_apart
#print axioms clause_gives_double_negation
#print axioms backward_of_excluded_middle
#print axioms double_negation_of_backward
#print axioms excluded_middle_of_backward
#print axioms backward_reading_iff_excluded_middle
#print axioms backward_in_completed_rules

#print axioms derived_law_on_sponsor
#print axioms excludes_fourth_case
#print axioms first_three_cases_admitted
#print axioms fourth_case_refuted
#print axioms conditional_is_not_exclusive
#print axioms no_rung_closes
#print axioms Derivability.chain
#print axioms Derivability.under
#print axioms Derivability.lob
#print axioms Derivability.lob_iff
#print axioms Derivability.underived_consistency

/-! ## The search, run -/

#eval (present demo ⟨7, rfl⟩).val
#eval (presentFromRefutation demo demo_refuted).val
'''

BUILD = WORK / "lean_check"
BUILD.mkdir(exist_ok=True)
(BUILD / "CompletenessOfValues.lean").write_text(LEAN_SOURCE, encoding="utf-8")
lean = shutil.which("lean")
code_, ver, _ = run([lean, "--version"])
print(ver.strip())
code_, out_, err_ = run([lean, "CompletenessOfValues.lean"], cwd=BUILD)
print("exit code:", code_)
print(out_.strip())
if err_.strip():
    print(err_.strip())

report = dict(re.findall(r"'CompletenessOfValues\.(\S+)' (depends on axioms: \[[^\]]*\]|does not depend on any axioms)", out_))
THREE = "depends on axioms: [propext, Classical.choice, Quot.sound]"
PROPEXT = "depends on axioms: [propext]"
NONE = "does not depend on any axioms"
expected = {
    "decided_in_value": THREE, "excluded_middle_from_selection": THREE, "value": THREE,
    "both_values": THREE, "underived_from_formation_data": THREE, "selection_fixes": NONE,
    "certificate_passage_iff_markov": NONE, "markov_in_completed_rules": THREE,
    "halts_of_exists": NONE, "step_wf": NONE, "searchFrom": NONE, "present": NONE,
    "presentFromRefutation": THREE, "counter_inscription_unfolds": NONE, "apart_unfolds": NONE,
    "false_has_no_case": NONE,
    "completed": THREE, "completed_presented": THREE, "completed_clause": THREE, "presented_of_apart": THREE,
    "clause_gives_double_negation": NONE, "backward_of_excluded_middle": NONE,
    "double_negation_of_backward": PROPEXT, "excluded_middle_of_backward": PROPEXT,
    "backward_reading_iff_excluded_middle": PROPEXT, "backward_in_completed_rules": THREE,
    "derived_law_on_sponsor": NONE, "excludes_fourth_case": NONE, "first_three_cases_admitted": NONE,
    "fourth_case_refuted": NONE, "conditional_is_not_exclusive": NONE, "no_rung_closes": NONE,
    "Derivability.chain": NONE, "Derivability.under": NONE, "Derivability.lob": NONE,
    "Derivability.lob_iff": NONE, "Derivability.underived_consistency": NONE,
}
ok = code_ == 0 and all(report.get(k) == v for k, v in expected.items())
runs = [l.strip() for l in out_.splitlines() if l.strip().isdigit()]
print()
print("compiled with exit code 0 and the axiom report is as the note states:", ok)
print("declarations reported:", len(report))
print("results of running the search:", runs)
RESULT["lean"] = {"version": ver.strip(), "ok": ok, "runs": runs, "declarations": len(report)}

Lean (version 4.33.1, x86_64-unknown-linux-gnu, commit 819816b2e0a3bf405af45ae5c7af2491d8f5bee6, Release)


exit code: 0
'CompletenessOfValues.nat_div_completed' depends on axioms: [propext]
'CompletenessOfValues.nat_mod_completed' depends on axioms: [propext]
'CompletenessOfValues.nat_sub_completed' does not depend on any axioms
'CompletenessOfValues.nat_pred_completed' does not depend on any axioms
'CompletenessOfValues.counter_inscription_unfolds' does not depend on any axioms
'CompletenessOfValues.apart_unfolds' does not depend on any axioms
'CompletenessOfValues.false_has_no_case' does not depend on any axioms
'CompletenessOfValues.decided_in_value' depends on axioms: [propext, Classical.choice, Quot.sound]
'CompletenessOfValues.excluded_middle_from_selection' depends on axioms: [propext, Classical.choice, Quot.sound]
'CompletenessOfValues.decided_in_value_from_selection' depends on axioms: [propext, Classical.choice, Quot.sound]
'CompletenessOfValues.value' depends on axioms: [propext, Classical.choice, Quot.sound]
'CompletenessOfValues.value_true_iff' depends on axioms: [propext, Clas

### The mark `noncomputable`

The function `value` of the note's Lean file assigns to every statement one of the two values. The cell compiles its definition with the mark `noncomputable` left off and prints the prover's answer.

In [21]:
(BUILD / "ValueWithoutMark.lean").write_text(
    "def value (a : Prop) : Bool := @decide a (Classical.propDecidable a)\n", encoding="utf-8")
code_, out_, err_ = run([lean, "ValueWithoutMark.lean"], cwd=BUILD)
print("exit code:", code_)
print((out_ + err_).strip())
RESULT["lean"]["value_without_mark_rejected"] = (code_ != 0 and "noncomputable" in out_ + err_)

exit code: 1
ValueWithoutMark.lean:1:4: error(lean.dependsOnNoncomputable): failed to compile definition, consider marking it as 'noncomputable' because it depends on 'Classical.propDecidable', which is 'noncomputable'


### Numerical values

The functional equation, in the form the library proves it, reads $\zeta(1-s)=2\,(2\pi)^{-s}\,\Gamma(s)\cos(\pi s/2)\,\zeta(s)$. At $s=2$ its right side is computed from $\zeta(2)=\pi^2/6$; the value is the value of the continuation at $-1$. The second value is the one the library's construction ascribes to the zeta function at its pole, $(\gamma-\log 4\pi)/2$. The third computation shows, at the point $s=\tfrac34$ of the strip, the growing partial sums of the series, the growing term $N^{1-s}/(1-s)$, and their difference, which settles.

In [22]:
zeta2 = math.pi ** 2 / 6
rhs = 2 * (2 * math.pi) ** (-2) * math.gamma(2) * math.cos(math.pi * 2 / 2) * zeta2
print("right side of the functional equation at s = 2 :", repr(rhs))
print("-1/12                                          :", repr(-1 / 12))
print("agreement to 1e-15:", abs(rhs + 1 / 12) < 1e-15)

EULER_GAMMA = 0.57721566490153286
pole_value = (EULER_GAMMA - math.log(4 * math.pi)) / 2
print()
print("(gamma - log(4 pi)) / 2 =", f"{pole_value:.10f}")

print()
s = 0.75
print(f"{'N':>10s}{'partial sum':>18s}{'N^(1-s)/(1-s)':>18s}{'difference':>18s}")
acc, n, rows = 0.0, 0, []
for N in (100, 10_000, 1_000_000):
    acc = math.fsum([acc] + [k ** (-s) for k in range(n + 1, N + 1)])
    n = N
    grow = N ** (1 - s) / (1 - s)
    rows.append((N, acc, grow, acc - grow))
    print(f"{N:>10,}{acc:>18.6f}{grow:>18.6f}{acc - grow:>18.6f}")
# the settled value, by the first terms of the Euler–Maclaurin correction at N = 1,000,000
N = 1_000_000
limit = acc - N ** (1 - s) / (1 - s) - N ** (-s) / 2 + s * N ** (-s - 1) / 12
print(f"settled value of the difference: {limit:.9f}")
RESULT.update(fe_rhs=rhs, pole_value=pole_value, strip_rows=rows, strip_limit=limit)

right side of the functional equation at s = 2 : -0.08333333333333333
-1/12                                          : -0.08333333333333333
agreement to 1e-15: True

(gamma - log(4 pi)) / 2 = -0.9769042910

         N       partial sum     N^(1-s)/(1-s)        difference
       100          9.223617         12.649111         -3.425494
    10,000         36.559215         40.000000         -3.440785


 1,000,000        123.049837        126.491106         -3.441270
settled value of the difference: -3.441285387


### The two integrals of Part 2d, and the direct bounds

The two lemmas of Part 2d conclude integrability from the values of two integrals. The cell computes both integrals numerically and compares them with the closed forms $1/a^{k+1}$ and $n^{-s}/(s(s-1))$. It also checks, on a grid, the two bounds by which integrability follows directly, with no appeal to the clause: $u^{k}e^{-au}/k!\le C_k\,e^{-au/2}$ with $C_k=(2k/(ae))^{k}/k!$ (and $C_0=1$), whose integral over the half-line is $2C_k/a$; and $|\max(y/n-1,0)\,y^{-s-1}|\le y^{-\sigma}/n$ for $y\ge1$, where $\sigma$ is the real part of $s$, whose integral over $y>1$ is $1/(n(\sigma-1))$.

In [23]:
from scipy import integrate
import numpy as np

print(f"{'k':>3s}{'a':>6s}{'integral':>20s}{'1/a^(k+1)':>20s}{'2 C_k / a':>16s}{'bound holds on grid':>22s}")
rows_k = []
for k in (0, 1, 5):
    for a in (0.5, 2.0):
        f = lambda u, k=k, a=a: u ** k * math.exp(-a * u) / math.factorial(k)
        val, err = integrate.quad(f, 0, math.inf, limit=400)
        Ck = 1.0 if k == 0 else (2 * k / (a * math.e)) ** k / math.factorial(k)
        grid = np.linspace(0, 80 / a, 4001)
        holds = bool(np.all(grid ** k * np.exp(-a * grid) / math.factorial(k) <= Ck * np.exp(-a * grid / 2) * (1 + 1e-12)))
        rows_k.append((k, a, val, a ** -(k + 1), 2 * Ck / a, holds))
        print(f"{k:>3d}{a:>6.1f}{val:>20.12f}{a ** -(k + 1):>20.12f}{2 * Ck / a:>16.6f}{str(holds):>22s}")

print()
print(f"{'n':>3s}{'s':>12s}{'integral':>34s}{'n^(-s)/(s(s-1))':>34s}{'1/(n(σ-1))':>14s}")
rows_n = []
for n in (1, 3):
    for s in (1.5 + 0j, 2.5 + 0j, 1.5 + 2j):
        g = lambda y, n=n, s=s: max(y / n - 1, 0.0) * y ** (-s - 1)
        re_, _ = integrate.quad(lambda y: g(y).real, n, math.inf, limit=800)
        im_, _ = integrate.quad(lambda y: g(y).imag, n, math.inf, limit=800)
        val = complex(re_, im_)
        closed = n ** (-s) / (s * (s - 1))
        rows_n.append((n, str(s), abs(val - closed)))
        print(f"{n:>3d}{str(s):>12s}{val:>34.10f}{closed:>34.10f}{1 / (n * (s.real - 1)):>14.6f}")
worst_k = max(abs(r[2] - r[3]) for r in rows_k)
worst_n = max(r[2] for r in rows_n)
print()
print(f"largest difference from the closed form: {worst_k:.2e} (first integral), {worst_n:.2e} (second integral)")
RESULT["integrals"] = {"worst_k": worst_k, "worst_n": worst_n, "bounds_hold": all(r[5] for r in rows_k)}

  k     a            integral           1/a^(k+1)       2 C_k / a   bound holds on grid
  0   0.5      2.000000000000      2.000000000000        4.000000                  True
  0   2.0      0.500000000000      0.500000000000        1.000000                  True
  1   0.5      4.000000000000      4.000000000000        5.886071                  True
  1   2.0      0.250000000000      0.250000000000        0.367879                  True
  5   0.5     64.000000000000     64.000000000000      718.714347                  True
  5   2.0      0.015625000000      0.015625000000        0.175467                  True

  n           s                          integral                   n^(-s)/(s(s-1))    1/(n(σ-1))
  1    (1.5+0j)        1.3333333333+0.0000000000j        1.3333333333-0.0000000000j      2.000000
  1    (2.5+0j)        0.2666666667+0.0000000000j        0.2666666667-0.0000000000j      0.666667
  1    (1.5+2j)       -0.1223529412-0.1505882353j       -0.1223529412-0.1505882353j      

## Part 6. The numbers that the note reports

In [24]:
fw, pl = RESULT["files_with"], RESULT["places"]
la, ro, ge, cs = RESULT["landau"], RESULT["route"], RESULT["geo"], RESULT["clause_steps"]
table = [
    ("Lean source files under lean/src", f"{RESULT['files']:,}"),
    ("lines", f"{RESULT['lines']:,}"),
    ("files containing the word sorry / admit / axiom / native_decide",
     f"{fw.get('sorry', 0)} / {fw.get('admit', 0)} / {fw.get('axiom', 0)} / {fw.get('native_decide', 0)}"),
    ("lines opening a declaration with the keyword axiom", f"{RESULT['decl'].get('axiom', 0)}"),
    ("files containing the string skipKernelTC", f"{RESULT['skip_kernel']}"),
    ("distinct options set with set_option", f"{RESULT['setopt_distinct']}"),
    ("lines declaring a name of the certified sentence", f"{RESULT['shadow']}"),
    ("theorems stated as False with no hypothesis", f"{RESULT['false_theorems']}"),
    ("definitions whose whole body is True or a function with the value True", f"{len(RESULT['true_defs'])}"),
    ("files containing the command run_cmd", f"{RESULT['run_cmd_files']}"),
    ("files containing by_contra (places)", f"{fw.get('by_contra', 0):,} ({pl.get('by_contra', 0):,})"),
    ("files containing exfalso (places)", f"{fw.get('exfalso', 0):,} ({pl.get('exfalso', 0):,})"),
    ("files containing absurd (places)", f"{fw.get('absurd', 0):,} ({pl.get('absurd', 0):,})"),
    ("files containing the string Classical. (places)", f"{fw.get('Classical.', 0):,} ({pl.get('Classical.', 0):,})"),
    ("files containing riemannZeta", f"{fw.get('riemannZeta', 0):,}"),
    ("files containing RiemannHypothesis", f"{fw.get('RiemannHypothesis', 0):,}"),
    ("files containing riemannZeta_one_sub", f"{fw.get('riemannZeta_one_sub', 0):,}"),
    ("files containing div_zero / inv_zero", f"{fw.get('div_zero', 0):,} / {fw.get('inv_zero', 0):,}"),
    ("files containing tsum_eq_zero_of_not_summable", f"{fw.get('tsum_eq_zero_of_not_summable', 0):,}"),
    ("files containing the string eulerProduct", f"{fw.get('eulerProduct', 0):,}"),
    ("files containing vonMangoldt", f"{fw.get('vonMangoldt', 0):,}"),
    ("lines opening a theorem or a lemma", f"{RESULT['decl'].get('theorem', 0) + RESULT['decl'].get('lemma', 0):,}"),
    ("links of the route / every link holds", f"{ro['links']} / {ro['all_hold']}"),
    ("folder LandauTheorems: files / lines", f"{la['files']} / {la['lines']:,}"),
    ("folder LandauTheorems: declarations / reached from the windowed criterion", f"{la['declarations']} / {la['reached']}"),
    ("reached declarations that call by_contra / by_cases", f"{len(la['by_contra_reached'])} / {la['by_cases_reached']}"),
    ("reached declarations that call integral_undef / of_integral_ne_zero / the clause for the derivative",
     f"{len(la['calls']['integral_undef']['reached'])} / {len(la['calls']['of_integral_ne_zero']['reached'])} / "
     f"{len(la['calls']['deriv_zero_of_not_differentiableAt']['reached'])}"),
    ("of_integral_ne_zero: places / files in the archive", f"{pl.get('of_integral_ne_zero', 0)} / {fw.get('of_integral_ne_zero', 0)}"),
    ("integral_undef: places / files in the archive", f"{pl.get('integral_undef', 0)} / {fw.get('integral_undef', 0)}"),
    ("anyGeometry: places / files in the archive", f"{pl.get('anyGeometry', 0)} / {fw.get('anyGeometry', 0)}"),
    ("anyGeometry: in geoCl anyGeometry / in OX or OAt / last argument of a lemma / its definition",
     f"{ge['where_any'].get('in geoCl anyGeometry', 0)} / "
     f"{ge['where_any'].get('in OX anyGeometry', 0) + ge['where_any'].get('in OAt anyGeometry', 0)} / "
     f"{ge['where_any'].get('the last argument of a lemma', 0)} / {ge['where_any'].get('its definition', 0)}"),
    ("the argument that follows geoCl: anyGeometry / the variable Γ / the binder of its definition",
     f"{ge['after_geoCl'].get('anyGeometry', 0)} / {ge['after_geoCl'].get('Γ', 0)} / {ge['after_geoCl'].get('(Γ : GeoP)', 0)}"),
    ("declarations whose stated type is GeoP", f"{len(ge['typed'])}"),
    ("declarations with the short name geoOf", f"{len(ge['named'])}"),
    ("the word geoOf: places / files; of them the term of type GeoP: places / files",
     f"{ge['geoOf_places'][0]} / {ge['geoOf_places'][1]}; {ge['member'][0]} / {ge['member'][1]}"),
    ("lemmas about that term: name, places, files",
     "; ".join(f"{k} {v[0]}, {v[1]}" for k, v in ge['lemmas_about'].items())),
    ("third argument of parametersOfExtension: the variable Γ / geoCl anyGeometry / geoCl Γ / the binder",
     f"{ge['entry']['parametersOfExtension'].get('Γ', 0)} / {ge['entry']['parametersOfExtension'].get('(geoCl anyGeometry)', 0)} / "
     f"{ge['entry']['parametersOfExtension'].get('(geoCl Γ)', 0)} / {ge['entry']['parametersOfExtension'].get('(Γ : GeoP)', 0)}"),
    ("'geoCl anyGeometry' in the statement of the consumer / in the supplier",
     f"{ge['consumer']['statement_geoCl_anyGeometry']} / {ge['supplier']['geoCl_anyGeometry']}"),
    ("'geoOf' in the statement of the consumer / in the supplier",
     f"{ge['consumer']['statement_geoOf']} / {ge['supplier']['geoOf']}"),
    ("the note's Lean file compiles and its axiom report is as stated", str(RESULT["lean"]["ok"])),
    ("declarations of the note's Lean file in the axiom report", str(RESULT["lean"]["declarations"])),
    ("the prover rejects the function value without the mark noncomputable", str(RESULT["lean"]["value_without_mark_rejected"])),
    ("right side of the functional equation at s = 2", f"{RESULT['fe_rhs']:.12f}"),
    ("the library's value of the zeta function at its pole", f"{RESULT['pole_value']:.10f}"),
    ("the two integrals agree with their closed forms to", f"{max(RESULT['integrals']['worst_k'], RESULT['integrals']['worst_n']):.1e}"),
]
for k, v in table:
    print(f"{k:104s}{v}")
_ = (WORK / "results.json").write_text(json.dumps(RESULT, indent=1, default=str))

Lean source files under lean/src                                                                        175,187
lines                                                                                                   17,374,215
files containing the word sorry / admit / axiom / native_decide                                         0 / 0 / 0 / 0
lines opening a declaration with the keyword axiom                                                      0
files containing the string skipKernelTC                                                                0
distinct options set with set_option                                                                    33
lines declaring a name of the certified sentence                                                        0
theorems stated as False with no hypothesis                                                             0
definitions whose whole body is True or a function with the value True                                  17
files containing 

## Part 7. The readings added in version 3

From version 3 on, the note looks each name up in a published list of the names in the exported proof of `Main.lean`, reads the last step of the windowed criterion field by field, and reads the proof of Siegel's theorem and the degenerate model of the archive's geometries. As in the earlier parts, the notebook prints statements and definitions from the archive; of proofs it prints line numbers, the names of commands and of record fields, and no proof text.

### 7a. The list of the names in the exported proof

The list is the file `results/used_decls.txt.gz` of the repository of I. Dixit, at the commit named below: one line for each declaration, constructor and recursor of the exported proof, with its kind and its full name.

In [25]:
import gzip, hashlib
LIST_REPO   = "https://github.com/ishaan2006/rh-lean-reproduction"
LIST_COMMIT = "a36105bec79a25ab7847441683fb693c5e7572b5"
LIST_SHA    = "e611381141a290cef77430625ee2aaf6a5a9d45b4f573a33f0b30c27dd304607"
gz = fetch(f"https://raw.githubusercontent.com/ishaan2006/rh-lean-reproduction/{LIST_COMMIT}/results/used_decls.txt.gz",
           WORK / "exported_proof" / "used_decls.txt.gz")
sha = hashlib.sha256(gz.read_bytes()).hexdigest()
print("repository:", LIST_REPO, "| commit:", LIST_COMMIT)
print("SHA-256 of the file:", sha)
print("as recorded in the note:", sha == LIST_SHA)
with gzip.open(gz, "rt", encoding="utf-8") as f:
    LIST = [l for l in f.read().split("\n") if l]
kinds = Counter(l.split(" ", 1)[0] for l in LIST)
print("lines:", f"{len(LIST):,}")
for k, v in kinds.most_common():
    print(f"  {k:8s}{v:>12,}")
BY_LAST = defaultdict(list)
for l in LIST:
    BY_LAST[l.split(" ", 1)[1].rsplit(".", 1)[-1]].append(l)

def in_export(short):
    "The lines of the list whose full name is the short name or ends in a dot followed by it."
    return BY_LAST.get(short, [])

RESULT["v3"] = {"list_lines": len(LIST), "list_kinds": dict(kinds), "list_sha_ok": sha == LIST_SHA}

repository: https://github.com/ishaan2006/rh-lean-reproduction | commit: a36105bec79a25ab7847441683fb693c5e7572b5
SHA-256 of the file: e611381141a290cef77430625ee2aaf6a5a9d45b4f573a33f0b30c27dd304607
as recorded in the note: True


lines: 1,484,733
  thm        1,246,110
  def          214,241
  ctor           8,342
  rec            7,972
  ind            7,970
  opaque            91
  quot               4
  axiom              3


### 7b. The two halves of the windowed criterion, and the lemma at which they meet

The record `PoleData`, the statements of the declarations of the last step, the fields of the record that each proof names, and the lines of the commands that open a refutation or a division into cases.

In [26]:
LT = SRC / "Development" / "AutomorphicArithmetic" / "DedekindLFunctions"
DECL_RX = r"\s*(?:@\[[^\]]*\]\s*)?(?:private |protected |noncomputable )*(?:theorem|lemma|def|structure)\s+(?:\S+\.)?"
STOP_RX = re.compile(r"(?:@\[[^\]]*\]\s*)?(?:private |protected |noncomputable )*(?:theorem|lemma|def|structure|end|namespace|section)\b")

def locate(path, name):
    L = path.read_text(encoding="utf-8", errors="replace").split("\n")
    idx = [i for i, l in enumerate(L) if re.match(DECL_RX + re.escape(name) + r"(?![\w'.])", l)]
    assert len(idx) == 1, (path.name, name, len(idx))
    i = idx[0]
    j = i + 1
    while j < len(L) and not STOP_RX.match(L[j]):
        j += 1
    return L, i, j

def stmt(path, name, max_lines=14, width=178):
    "Print the statement of a declaration: its lines up to the sign := that opens the proof."
    L, i, j = locate(path, name)
    for k, l in enumerate(L[i:min(j, i + max_lines)]):
        cut = l.find(":=")
        print(f"{path.name}:{i + 1 + k}: {(l if cut < 0 else l[:cut + 2])[:width]}")
        if cut >= 0:
            break
    print()

def proof_facts(path, name, record="hP"):
    "Of a proof: its line span, the fields of the record that it names, and the lines of three commands."
    L, i, j = locate(path, name)
    text = "\n".join(L[i:j])
    fields = sorted(set(re.findall(r"\b" + record + r"\.(\w+)", text)))
    cmds = {c: [i + 1 + k for k, l in enumerate(L[i:j]) if re.search(r"\b" + c + r"\b", l)]
            for c in ("by_contra", "by_cases", "eq_empty_or_nonempty")}
    return {"file": path.name, "lines": (i + 1, j), "fields": fields, "commands": cmds, "text": text}

show(LT / "LandauZeroFreeCriterion.lean", r"^structure PoleData", after=8)
LW = LT / "LandauTheorems"
WHERE = {"landau_window": "landauWindow.lean", "landau_window_key": "landauWindow.lean",
         "landau_window_contra": "landauWindow.lean", "landau_coeff": "LandauCoeff.lean",
         "landau_dichotomy": "LandauDichotomy.lean", "landau_nearest": "LandauNearest.lean",
         "landau_finite": "LandauFinite.lean", "landau_pole_asym": "LandauPoleAsym.lean",
         "logDeriv_local": "LogDerivLocal.lean", "landau_growth_contra": "LandauGrowthContra.lean"}
for n, f in WHERE.items():
    stmt(LW / f, n)
stmt(LW / "poleData.lean", "poleData")

LandauZeroFreeCriterion.lean:15: structure PoleData (g : ℝ → ℝ) (Φ : ℂ → ℂ) (Z : Set ℂ) : Prop where
LandauZeroFreeCriterion.lean:16:   meas : Measurable g
LandauZeroFreeCriterion.lean:17:   growth : ∃ C : ℝ, ∀ u : ℝ, 0 ≤ u → |g u| ≤ C * Real.exp (2 * u)
LandauZeroFreeCriterion.lean:18:   laplace : ∀ s : ℂ, 2 < s.re → Φ s = ∫ u in Ioi (0 : ℝ), (g u : ℂ) * cexp (-s * u)
LandauZeroFreeCriterion.lean:19:   pos : ∀ ρ ∈ Z, 0 < ρ.re
LandauZeroFreeCriterion.lean:20:   diff : DifferentiableOn ℂ Φ ({s : ℂ | 0 < s.re} \ Z)
LandauZeroFreeCriterion.lean:21:   iso : ∀ z : ℂ, 0 < z.re → ∀ᶠ w in 𝓝[≠] z, w ∉ Z
LandauZeroFreeCriterion.lean:22:   pole : ∀ ρ ∈ Z, ∃ c : ℂ, c ≠ 0 ∧ (ρ.im = 0 → 0 < c.re ∧ c.im = 0) ∧
LandauZeroFreeCriterion.lean:23:     ∃ h : ℂ → ℂ, AnalyticAt ℂ h ρ ∧ ∀ᶠ s in 𝓝[≠] ρ, Φ s = c / (s - ρ) + h s

landauWindow.lean:83: theorem landau_window {g : ℝ → ℝ} {Φ : ℂ → ℂ} {Z : Set ℂ} (hP : PoleData g Φ Z) {θ δ : ℝ}
landauWindow.lean:84:     (hθ : 0 < θ) (hθ1 : θ < 1) (hδ : 0 < δ) (hW : W

In [27]:
facts = {n: proof_facts(LW / f, n) for n, f in WHERE.items()}
facts["poleData"] = proof_facts(LW / "poleData.lean", "poleData")
print(f"{'declaration':24s}{'lines':>12s}   fields of the record named in the proof")
for n in ("landau_window_key", "landau_coeff", "landau_window"):
    p = facts[n]
    print(f"{n:24s}{p['lines'][0]:>6d}-{p['lines'][1]:<5d}   {', '.join(p['fields']) or '(none)'}")
key, coeff, win = facts["landau_window_key"], facts["landau_coeff"], facts["landau_window"]
half_one = set(key["fields"]) | set(coeff["fields"])
print()
print("the first half (landau_window_key with landau_coeff) names only meas, growth and laplace:",
      half_one <= {"meas", "growth", "laplace"} and {"meas", "growth", "laplace"} <= half_one)
print("landau_window_key calls landau_coeff:", "landau_coeff" in key["text"])
print("landau_window names diff, iso and pole:", {"diff", "iso", "pole"} <= set(win["fields"]))
print("landau_window calls, in order of first occurrence:",
      [c for c in sorted(("landau_window_key", "landau_dichotomy", "landau_nearest", "landau_pole_asym", "landau_cauchy",
                          "landau_window_contra"), key=lambda c: win["text"].find(c)) if c in win["text"]])
print("the proof of poleData reaches logDeriv_local through poleData_abstract:",
      "logDeriv_local" in (LW / "poleData.lean").read_text(encoding="utf-8"))
print()
print(f"{'declaration':24s} lines of by_contra / by_cases / eq_empty_or_nonempty")
for n in ("landau_window", "landau_dichotomy", "landau_nearest", "landau_finite", "landau_window_contra"):
    c = facts[n]["commands"]
    print(f"{n:24s} {c['by_contra']} / {c['by_cases']} / {c['eq_empty_or_nonempty']}")
print()
print("landau_window_contra is stated for real numbers and concludes False:",
      ": False" in facts["landau_window_contra"]["text"].split(":=")[0])
RESULT["v3"]["junction"] = {
    "first_half_fields": sorted(half_one), "window_fields": win["fields"],
    "by_contra_in_landau_window": len(win["commands"]["by_contra"]),
    "by_cases_in_dichotomy": len(facts["landau_dichotomy"]["commands"]["by_cases"]),
    "empty_or_nonempty_in_dichotomy": len(facts["landau_dichotomy"]["commands"]["eq_empty_or_nonempty"]),
    "by_contra_in_nearest": len(facts["landau_nearest"]["commands"]["by_contra"]),
}

declaration                    lines   fields of the record named in the proof
landau_window_key           40-74      growth, meas
landau_coeff                10-29      growth, laplace, meas
landau_window               83-167     diff, iso, pole

the first half (landau_window_key with landau_coeff) names only meas, growth and laplace: True
landau_window_key calls landau_coeff: True
landau_window names diff, iso and pole: True
landau_window calls, in order of first occurrence: ['landau_window_key', 'landau_dichotomy', 'landau_nearest', 'landau_pole_asym', 'landau_cauchy', 'landau_window_contra']
the proof of poleData reaches logDeriv_local through poleData_abstract: True

declaration              lines of by_contra / by_cases / eq_empty_or_nonempty
landau_window            [86] / [] / []
landau_dichotomy         [84] / [47] / [32, 70]
landau_nearest           [107] / [] / [90]
landau_finite            [] / [24] / []
landau_window_contra     [] / [] / []

landau_window_contra is stated 

### 7c. The division in the proof of Siegel's theorem

The statement, the place of the command `by_cases` in its proof with the names that occur in the statement on which it divides, the place at which the first case uses the supposed zero, the theorem of `Main.lean` that refutes that statement, and the opening of the statement of the bound on windows.

In [28]:
SZ = SRC / "Formalism" / "LandauZeroFreeBounds"
show(SZ / "SiegelWalfiszAP.lean", r"^def SiegelLower", after=2)
sg = proof_facts(SZ / "ExceptionalZeroBounds" / "SiegelOf.lean", "siegel_theorem_of", record="h")
L, i, j = locate(SZ / "ExceptionalZeroBounds" / "SiegelOf.lean", "siegel_theorem_of")
bc = [k for k in range(i, j) if "by_cases" in L[k]]
div = " ".join(L[bc[0]:bc[0] + 2])
print("siegel_theorem_of: lines", (i + 1, j), "| commands by_cases:", len(bc), "| at line", bc[0] + 1)
print("the statement on which it divides contains")
for label, token in (("an existence sign", "∃"), ("a real number β", "(β : ℝ)"), ("a character whose square is trivial", "χ₁ ^ 2 = 1"),
                     ("apart from the trivial character", "χ₁ ≠ 1"), ("β below 1", "β < 1"),
                     ("the L-function is 0 at β", "DirichletCharacter.LFunction χ₁ β = 0")):
    print(f"   {label:38s}", token in div)
uses = [k + 1 for k in range(bc[0], j) if re.search(r"\bhzero\b", L[k])]
print("lines after the division at which the supposed zero (the name hzero) occurs:", uses)
print()
show(SRC / "Statements.lean", r"^def DirichletLNonvanishingRightOfHalf", after=2)
show(SRC / "Main.lean", r"LFunction_ne_zero_of_half_lt_re")
stmt(SRC / "Development" / "AutomorphicArithmetic" / "PrimitiveCosets" / "CoreGm.lean", "core_gm", max_lines=6)
callers_exc = []
for d in ("Development/AutomorphicArithmetic/MeanSquareBounds", "Formalism/GL3MeanSquares"):
    for p in sorted((SRC / d).rglob("*.lean")):
        t_ = p.read_text(encoding="utf-8", errors="replace")
        n_ = len(re.findall(r"ZeroFreeDirichlet\.excBound\b", t_))
        if n_:
            callers_exc.append((str(p.relative_to(SRC)), n_))
print("files of the two folders searched in which the name ZeroFreeDirichlet.excBound occurs:")
for p, n_ in callers_exc:
    print("  ", n_, p)
RESULT["v3"]["siegel"] = {"by_cases": len(bc), "line": bc[0] + 1, "hzero_lines": uses, "excBound_files": len(callers_exc)}

SiegelWalfiszAP.lean:29: def SiegelLower : Prop :=
SiegelWalfiszAP.lean:30:   ∀ ε : ℝ, 0 < ε → ∃ c : ℝ, 0 < c ∧ ∀ (q : ℕ) [NeZero q] (χ : DirichletCharacter ℂ q), χ ^ 2 = 1 → χ ≠ 1 →
SiegelWalfiszAP.lean:31:     c * (q : ℝ) ^ (-ε) ≤ ‖DirichletCharacter.LFunction χ 1‖

siegel_theorem_of: lines (9, 65) | commands by_cases: 1 | at line 30
the statement on which it divides contains
   an existence sign                      True
   a real number β                        True
   a character whose square is trivial    True
   apart from the trivial character       True
   β below 1                              True
   the L-function is 0 at β               True
lines after the division at which the supposed zero (the name hzero) occurs: [33, 37]

Statements.lean:6: def DirichletLNonvanishingRightOfHalf : Prop :=
Statements.lean:7:   ∀ (N : ℕ) [NeZero N] (χ : DirichletCharacter ℂ N) (s : ℂ),
Statements.lean:8:     1 / 2 < s.re → (χ ≠ 1 ∨ s ≠ 1) → DirichletCharacter.LFunction χ s ≠ 0

Main.lean

### 7d. The degenerate model of the archive's geometries

The definition of `geoDegenerate`, the opening of `divXFl`, and the statements of `geoOver_stages`, `divEx_prov`, `GeoTied` and `FormsTieX`.

In [29]:
LG = SRC / "Formalism" / "LocalToGlobal" / "LatticeGeometry"
show(LG / "Inhabited.lean", r"def geoDegenerate\b", after=24, width=150)
show(LG / "Inhabited.lean", r"^def divXFl\b", after=3)
TP = SRC / "Development" / "AbelianVarietyTheory" / "Proofs" / "IwasawaDecomposition" / "TupPinsOfMuIsW.lean"
stmt(TP, "geoOver_stages")
stmt(TP, "divEx_prov")
show(LG / "GeoTied.lean", r"^def GeoTied\b", after=3)
show(LG / "GeoTiedP.lean", r"^def FormsTieX\b", after=4)
gd = (LG / "Inhabited.lean").read_text(encoding="utf-8")
k0 = gd.index("def geoDegenerate ")
blk = gd[k0:k0 + 1800]
deg = {f: bool(re.search(r"\b" + f + r" := fun [_ ]*=> True", blk)) for f in ("Eff", "nef", "big", "bpf")}
print("in geoDegenerate, defined as True at every argument:", deg)
print("Var := ℕ:", "Var := ℕ" in blk, "| H0 is the whole space (⊤):", "H0 := fun _ _ => ⊤" in blk)
RESULT["v3"]["degenerate"] = deg

Inhabited.lean:241: @[reducible] def geoDegenerate (F : Type) [Field F] [NumberField F] [StarRing F] (Tau : Type) [Fintype Tau] : DV.Geo F where
Inhabited.lean:242:   Var := ℕ
Inhabited.lean:243:   dim := fun N => Formalism.DegreeFormula.dimFl Tau N
Inhabited.lean:244:   Fn := fun _ => F
Inhabited.lean:245:   fnField := fun _ => inferInstanceAs (Field F)
Inhabited.lean:246:   fnAlg := fun _ => inferInstanceAs (Algebra F F)
Inhabited.lean:247:   Dv := fun N => Tau → Fin N → ℤ
Inhabited.lean:248:   dvGrp := fun _ => inferInstance
Inhabited.lean:249:   Eff := fun _ _ => True
Inhabited.lean:250:   eff_zero := fun _ => trivial
Inhabited.lean:251:   eff_add := fun _ _ _ _ _ => trivial
Inhabited.lean:252:   H0 := fun _ _ => ⊤
Inhabited.lean:253:   degreeZeroCohomology_finiteDimensional := fun _ _ => inferInstance
Inhabited.lean:254:   mul_mem_riemannRochSpace_add := fun _ _ _ _ _ _ _ => Submodule.mem_top
Inhabited.lean:255:   riemannRoch_mono := fun _ _ _ _ => le_refl _
Inhabited.lean:256:   

### 7e. Names in the exported proof, and names not in it

A name is *in the exported proof* when some line of the list has it as its full name or as the last component of its full name. For the names that the note says are not in the exported proof the cell also counts, by a search of the whole archive, the declarations of each name in its text.

In [30]:
IN_NAMES = {
 "the last step of the criterion": ["landau_window", "landau_window_key", "landau_coeff", "landau_window_contra", "landau_growth_contra",
                                    "landau_dichotomy", "landau_nearest", "landau_finite", "landau_pole_asym", "logDeriv_local",
                                    "poleData", "poleData_abstract", "zeroFree_of_rpow_window", "lemmaL"],
 "three steps that call a clause": ["landau_tails_integrable", "mellin_Gchi_of_ne_one_int", "deriv_conj_comp_conj"],
 "Siegel's theorem and the bound": ["siegel_theorem_of", "siegelLower", "excBound", "twistK", "core_gm"],
 "the condition that every object satisfies": ["anyGeometry", "geoCl", "parametersOfExtension",
                                    "hypotheses_nonempty_of_unramified_exponents_alt", "nonempty_hyps_definite_side_of_closed_bound"],
 "the degenerate model": ["geoDegenerate", "divXFl", "geoOver_stages", "divEx_prov"],
 "named for arithmetic geometry": ["ArithHilbertSamuel", "KawamataViehweg", "hodge_bundle_isNef_isBig_of_weight",
                                    "chowla_selberg_eta", "topCovolG_alt_alt"],
}
OUT_NAMES = {
 "the condition that asks for a geometric witness": ["geoOf", "IsGeoOf", "HasTrueSurf"],
 "consequences of the tie condition, and conditions on a family over a variety": ["GeoTied", "GeoTiedM", "FormsTieX", "formsTieX_of_cusp", "MumfordU", "CurveHTDom",
                                    "chern_linRoot_pow", "shimura_sheaf_of_indefinite_subspace"],
}
ok_in, ok_out, ok_declared = True, True, True
ALL_OUT = [n for names in OUT_NAMES.values() for n in names]
pat = (r"^\s*(@\[[^]]*\]\s*)?(private |protected |noncomputable )*(theorem|lemma|def|abbrev|structure|class|inductive)\s+(\S+\.)?("
       + "|".join(ALL_OUT) + r")([^A-Za-z0-9_'.]|$)")
code_, out_, err_ = run(["grep", "-rnE", pat, "--include=*.lean", str(SRC)], timeout=1800)
DECLARED = Counter()
for line in out_.splitlines():
    m = re.search(r"(?:theorem|lemma|def|abbrev|structure|class|inductive)\s+(?:\S+\.)?(" + "|".join(ALL_OUT) + r")(?![A-Za-z0-9_'.])", line)
    if m:
        DECLARED[m.group(1)] += 1
for group, names in IN_NAMES.items():
    print(group)
    for n in names:
        found = in_export(n)
        ok_in = ok_in and len(found) > 0
        print(f"   {n:52s}{len(found):>3d}   {found[0] if found else '(not in the list)'}")
print()
for group, names in OUT_NAMES.items():
    print(group)
    for n in names:
        found = in_export(n)
        ok_out = ok_out and len(found) == 0
        places = DECLARED.get(n, 0)
        ok_declared = ok_declared and places > 0
        print(f"   {n:44s} lines of the list: {len(found)}   declarations of the name in the text of the archive: {places}")
print()
print("every name of the first table is in the exported proof:", ok_in)
print("no name of the second table is in the exported proof:", ok_out)
print("every name of the second table is declared in the text of the archive:", ok_declared)
RESULT["v3"]["names_in"] = ok_in
RESULT["v3"]["names_out"] = ok_out
RESULT["v3"]["anyGeometry_line"] = in_export("anyGeometry")

the last step of the criterion
   landau_window                                         1   thm Deduction.landau_window
   landau_window_key                                     1   thm Deduction.landau_window_key
   landau_coeff                                          1   thm Deduction.landau_coeff
   landau_window_contra                                  1   thm Deduction.landau_window_contra
   landau_growth_contra                                  1   thm Deduction.landau_growth_contra
   landau_dichotomy                                      1   thm Deduction.landau_dichotomy
   landau_nearest                                        1   thm Deduction.landau_nearest
   landau_finite                                         1   thm Deduction.landau_finite
   landau_pole_asym                                      1   thm Deduction.landau_pole_asym
   logDeriv_local                                        3   thm Deduction.logDeriv_local
   poleData                                           

### 7f. The four combinations

`L` is "the law holds at every whole number" and `K` is "the rules are consistent". What the certified derivation establishes outright is the conditional "if not `L`, then not `K`". The cell prints its value at each of the four combinations.

In [31]:
rows = []
for L_, K_ in ((True, True), (True, False), (False, False), (False, True)):
    cond = (not (not L_)) or (not K_)          # "if not L, then not K"
    rows.append((L_, K_, cond))
    print(f"L = {str(L_):5s}  K = {str(K_):5s}  'if not L, then not K' = {cond}")
excluded = [(a, b) for a, b, c in rows if not c]
print()
print("combinations excluded by the conditional:", excluded)
print("the combination in which the law holds and the rules derive a clash is admitted:", (True, False, True) in rows)
RESULT["v3"]["four_cases_excluded"] = excluded

L = True   K = True   'if not L, then not K' = True
L = True   K = False  'if not L, then not K' = True
L = False  K = False  'if not L, then not K' = True
L = False  K = True   'if not L, then not K' = False

combinations excluded by the conditional: [(False, True)]
the combination in which the law holds and the rules derive a clash is admitted: True


### 7g. The verification record on 23 November 2026

The record of A. V. Sutherland is a web page outside the hosts that this notebook reaches. The cell prints what was read from it on 23 November 2026 at 01:15 UTC, for comparison with the reading of 22 November reported in Proposition "The state of the certification".

In [32]:
record_23 = [
 ("header of the page", "Page last updated 2026-11-22 02:25 UTC (unchanged)"),
 ("latest entry", "22 November 2026, 04:16 UTC (unchanged)"),
 ("prover's kernel, default mode, on the exported proof", "running, no result"),
 ("prover's kernel, paranoid mode, on the exported proof", "running, no result"),
 ("comparison tool, end to end", "running; comparison of statements and check of axioms passed"),
 ("lean4lean replay of the compiled modules", "passed, 175,187 of 175,187"),
 ("nanoda, con-leche and con-ron (current sources) on the exported proof", "accept"),
 ("con-leche and con-ron as distributed with version 4.36.0", "declined: a reflexive inductive type is unsupported"),
 ("declarations of the exported proof, by con-leche", "1,468,418"),
 ("axioms of the five theorems", "propext, Classical.choice, Quot.sound"),
]
for k, v in record_23:
    print(f"{k:72s}{v}")
RESULT["v3"]["record_read"] = "2026-11-23 01:15 UTC"

header of the page                                                      Page last updated 2026-11-22 02:25 UTC (unchanged)
latest entry                                                            22 November 2026, 04:16 UTC (unchanged)
prover's kernel, default mode, on the exported proof                    running, no result
prover's kernel, paranoid mode, on the exported proof                   running, no result
comparison tool, end to end                                             running; comparison of statements and check of axioms passed
lean4lean replay of the compiled modules                                passed, 175,187 of 175,187
nanoda, con-leche and con-ron (current sources) on the exported proof   accept
con-leche and con-ron as distributed with version 4.36.0                declined: a reflexive inductive type is unsupported
declarations of the exported proof, by con-leche                        1,468,418
axioms of the five theorems                                       

### 7h. The tie condition that is in the exported proof (added in version 6)

`GeoTiedP` is the archive's statement that a divisor record is tied to a family over its geometry, with four further conditions; `GeoTied` is the same statement with the four conditions left out. The cell looks the names up as whole names (the full name of a line, or a suffix of it that begins after a dot), prints the two definitions and the type of the family `famFlJ`, and reports, for the proof of `geoOver_stages`, which of six names occur in it. Of that proof it prints no text.

In [33]:
FULL = [l.split(" ", 1)[1] for l in LIST]
FULLSET = set(FULL)
SUFFIX = defaultdict(int)
for f in FULL:
    parts = f.split(".")
    for k in range(len(parts)):
        SUFFIX[".".join(parts[k:])] += 1
def whole(name):
    "Number of lines of the list whose full name is the name or ends in a dot followed by it."
    return SUFFIX.get(name, 0)
TIE_IN  = ["GeoTiedP", "GeoOver", "geoOver_stages", "geoTiedP_divTr", "divXFl", "famFlJ", "tiedX_fl", "isoInv_famFlJ",
           "levInTie_famFlJ", "geoDegenerate", "GeoOfMatch", "geoOfMatch_holds"]
TIE_OUT = ["GeoTied", "GeoTiedM", "FormsTieX", "GeoTiedP.geoTied", "GeoTiedP.formsTieX", "GeoOver.mono", "GeoTiedPL", "geoTiedP_satisfiable",
           "geoOf", "IsGeoOf", "HasTrueSurf"]
for n in TIE_IN:
    print(f"   in the list      {n:26s}{whole(n):>3d}")
for n in TIE_OUT:
    print(f"   not in the list  {n:26s}{whole(n):>3d}")
tie_in_ok, tie_out_ok = all(whole(n) > 0 for n in TIE_IN), all(whole(n) == 0 for n in TIE_OUT)
print("every name of the first group is in the exported proof:", tie_in_ok, "| no name of the second group is:", tie_out_ok)
print()
show(LG / "GeoTied.lean", r"^def GeoTied\b", after=3)
show(LG / "GeoTiedP.lean", r"^def GeoTiedP\b", after=4)
show(LG / "GeoTiedP.lean", r"^def famFlJ\b", after=2)
L, i, j = locate(TP, "geoOver_stages")
body = "\n".join(L[i:j])
names6 = ["divXFl", "famFlJ", "tiedX_fl", "isoInv_famFlJ", "levInTie_famFlJ", "trivial"]
occ = {n: bool(re.search(r"\b" + n + r"\b", body)) for n in names6}
print()
print("names that occur in the proof of geoOver_stages:", occ)
print("the condition on levels supplied there is the one that holds of every level:", "fun _ _ => True" in body)
gt = (LG / "GeoTiedP.lean").read_text(encoding="utf-8")
k0 = gt.index("def famFlJ "); fam_over_degenerate = "ShimFam D (geoDegenerate" in gt[k0:k0 + 700]
print("famFlJ is a family over geoDegenerate:", fam_over_degenerate)
OI = SRC / "Development" / "AutomorphicArithmetic" / "AutomorphicVarieties" / "ModularityArgument" / "TraceFormula" / "OrbitalIntegrals" / "GeoTransport.lean"
show(OI, r"^def GeoOfMatch\b", after=1, width=150)
c_, o_, e_ = run(["grep", "-rnE", r"\bGeoOver\b", "--include=*.lean", str(SRC)], timeout=1800)
go = [l for l in o_.splitlines()]
print()
print("lines of the archive that name GeoOver:", len(go))
for l in go:
    f, n, t = l.split(":", 2)
    print("  ", Path(f).name + ":" + n + ":", t.strip().split(":=")[0][:150])
RESULT["v3"]["tie"] = {"in": tie_in_ok, "out": tie_out_ok, "names_in_geoOver_stages": occ, "famFlJ_over_degenerate": fam_over_degenerate}

   in the list      GeoTiedP                    1
   in the list      GeoOver                     1
   in the list      geoOver_stages              1
   in the list      geoTiedP_divTr              1
   in the list      divXFl                      1
   in the list      famFlJ                      1
   in the list      tiedX_fl                    1
   in the list      isoInv_famFlJ               1
   in the list      levInTie_famFlJ             1
   in the list      geoDegenerate               1
   in the list      GeoOfMatch                  1
   in the list      geoOfMatch_holds            1
   not in the list  GeoTied                     0
   not in the list  GeoTiedM                    0
   not in the list  FormsTieX                   0
   not in the list  GeoTiedP.geoTied            0
   not in the list  GeoTiedP.formsTieX          0
   not in the list  GeoOver.mono                0
   not in the list  GeoTiedPL                   0
   not in the list  geoTiedP_satisfiable        0



lines of the archive that name GeoOver: 3
   TupPinsOfMuIsW.lean:155: theorem geoOver_stages (X : Stages) (X₀ : DivX X.toD7) : GeoOver (fun _ => True) X₀ 
   GeoTiedPGraft.lean:45: def GeoOver (Q : DivX D → Prop) (X₀ : DivX D) : Prop 
   GeoTiedPGraft.lean:48: theorem GeoOver.mono {Q Q' : DivX D → Prop} (h : ∀ X, Q X → Q' X) {X₀ : DivX D} (g : GeoOver Q X₀) : GeoOver Q' X₀ 


### The numbers of Part 7

In [34]:
v3 = RESULT["v3"]
table7 = [
 ("lines of the list of the exported proof", f"{v3['list_lines']:,}"),
 ("of which theorems / definitions / axioms", f"{v3['list_kinds'].get('thm', 0):,} / {v3['list_kinds'].get('def', 0):,} / {v3['list_kinds'].get('axiom', 0)}"),
 ("the SHA-256 value of the list is the one recorded", str(v3["list_sha_ok"])),
 ("fields of PoleData named in the first half of the criterion", ", ".join(v3["junction"]["first_half_fields"])),
 ("commands by_contra in landau_window", str(v3["junction"]["by_contra_in_landau_window"])),
 ("commands by_cases / divisions on an empty finite set in landau_dichotomy", f"{v3['junction']['by_cases_in_dichotomy']} / {v3['junction']['empty_or_nonempty_in_dichotomy']}"),
 ("commands by_cases in siegel_theorem_of, and its line", f"{v3['siegel']['by_cases']} / {v3['siegel']['line']}"),
 ("every name said to be in the exported proof is in the list", str(v3["names_in"])),
 ("no name said to be absent is in the list", str(v3["names_out"])),
 ("combinations excluded by the conditional (L, K)", str(v3["four_cases_excluded"])),
 ("GeoTiedP and the model that fills it are in the list; GeoTied and FormsTieX are not", f"{v3['tie']['in']} / {v3['tie']['out']}"),
]
for k, v in table7:
    print(f"{k:80s}{v}")
_ = (WORK / "results.json").write_text(json.dumps(RESULT, indent=1, default=str))

lines of the list of the exported proof                                         1,484,733
of which theorems / definitions / axioms                                        1,246,110 / 214,241 / 3
the SHA-256 value of the list is the one recorded                               True
fields of PoleData named in the first half of the criterion                     growth, laplace, meas
commands by_contra in landau_window                                             1
commands by_cases / divisions on an empty finite set in landau_dichotomy        1 / 2
commands by_cases in siegel_theorem_of, and its line                            1 / 30
every name said to be in the exported proof is in the list                      True
no name said to be absent is in the list                                        True
combinations excluded by the conditional (L, K)                                 [(False, True)]
GeoTiedP and the model that fills it are in the list; GeoTied and FormsTieX are notTrue / True
